<script async src="https://www.googletagmanager.com/gtag/js?id=UA-59152712-8"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());

  gtag('config', 'UA-59152712-8');
</script>

# Example Implementation of GRHayL: Replicating the IllinoisGRMHD code using GRHayL

## Author: Terrence Pierre Jacques
### Formatting improvements courtesy Brandon Clark

## This notebook will utilize the General Relativistic Hydrodynamic Library (GRHayL) to recreate the results of the IllinoisGRMHD code, for a suite of GRMHD shock tests.

**Notebook Status:** <font color ="red"><b> Not validated </b></font>

<!-- **Validation Notes:** This module has been validated to converge at the expected order to the exact solution (see [plot](#convergence) at bottom). -->

<!-- ### NRPy+ Source Code for this module: 
* [ScalarWave/ScalarWaveCurvilinear_RHSs.py](../edit/ScalarWave/ScalarWaveCurvilinear_RHSs.py) [\[**tutorial**\]](Tutorial-ScalarWaveCurvilinear.ipynb) Generates the right-hand side for the Scalar Wave Equation in curvilinear coordinates
* [ScalarWave/InitialData.py](../edit/ScalarWave/InitialData.py) [\[**tutorial**\]](Tutorial-ScalarWave.ipynb) Generating C code for either plane wave or spherical Gaussian initial data for the scalar wave equation  -->

## Introduction:
GRHayL is code base that aims to provide robust and well tested codes for evolving relativistic fluids. Towards this end, GRHayL currently supports reconstruction, flux/source term calculations, hybrid and tabulated equations of state, and conservative-to-primitive solvers

The entire algorithm is outlined below, with NRPy+-based components highlighted in <font color='green'>green</font>.

1. Allocate memory for gridfunctions, including temporary storage for the RK4 time integration.
1. <font color='green'>Set gridfunction values to initial data.</font>
1. Evolve the system forward in time using RK4 time integration. At each RK4 substep, do the following:
    1. <font color='green'>GRMHD RHS expressions.</font>
    1. Apply conservatives-to-primitives routines
    1. Apply boundary conditions.

<a id='toc'></a>

# Table of Contents
$$\label{toc}$$

This notebook is organized as follows

1. [Step 1](#writec): Generate C code to solve the scalar wave equation in curvilinear coordinates
    1. [Step 1.a](#id_rhss): C code generation: Initial data and scalar wave right-hand-sides
    1. [Step 1.b](#OuterBoundaryConditions): C code generation: Boundary condition driver
    1. [Step 1.c](#cparams_rfm_and_domainsize): Generate Cparameters files; set reference metric parameters, including `domain_size`
    1. [Step 1.d](#cfl): C code generation: Finding the minimum proper distance between grid points, needed for [CFL](https://en.wikipedia.org/w/index.php?title=Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition&oldid=806430673)-limited timestep
1. [Step 2](#mainc): The C code `main()` function for `ScalarWaveCurvilinear_Playground`
1. [Step 3](#compileexec): Compile generated C codes & solve the scalar wave equation
1. [Step 4](#convergence): Code validation: Plot the numerical error, and confirm that it converges to zero at expected rate with increasing numerical resolution (sampling)
1. [Step 5](#latex_pdf_output): Output this notebook to $\LaTeX$-formatted PDF file

<a id='writec'></a>

# Step 1: Using NRPy+ to generate necessary C codes to supplement codes in GRHayL to solve the equations of GRMHD \[Back to [top](#toc)\]
$$\label{writec}$$

<a id='id_rhss'></a>

## Step 1.a: C code generation: Initial data and GRMHD RHSs \[Back to [top](#toc)\]
$$\label{id_rhss}$$

To evolve the GRMHD equations in time, we need to specify initial data, that we may march forward using the method of lines approach (MoL), and right-hand-sides (RHSs) to dictate how these data are evolved. 

We begin by importing some key NRPy+ python modules. We then construct C codes to supply initial data to our grid functions. These initial data are 1D and 2D shock tests, and all options are compiled so that at run time we may choose which initial data set we wish to evolve test. The tests ar stored in the ShockTests directory within NRPy+.

In [1]:
import shutil, os, sys                # Standard Python modules for multiplatform OS-level functions
# nrpy_dir_path = os.path.join("../../GRHayL/Flux_Source/nrpy")
# if nrpy_dir_path not in sys.path:
#     sys.path.append(nrpy_dir_path)

    # Step P1: Import needed NRPy+ core modules:
from outputC import lhrh, outCfunction, add_to_Cfunction_dict  # NRPy+: Core C code output module
import finite_difference as fin       # NRPy+: Finite difference C code generation module
import NRPy_param_funcs as par        # NRPy+: Parameter interface
import grid as gri                    # NRPy+: Functions having to do with numerical grids
import reference_metric as rfm        # NRPy+: Reference metric support
import indexedexp as ixp         # NRPy+: Symbolic indexed expression (e.g., tensors, vectors, etc.) support
import cmdline_helper as cmd          # NRPy+: Multi-platform Python command-line interface
import sympy as sp

# Step P2: Create C code output directory:
Ccodesrootdir = os.path.join("ghl_NRPy_IGM_Ccodes")
# First remove C code output directory if it exists
# Courtesy https://stackoverflow.com/questions/303200/how-do-i-remove-delete-a-folder-that-is-not-empty
shutil.rmtree(Ccodesrootdir, ignore_errors=True)
# Then create a fresh directory
cmd.mkdir(Ccodesrootdir)

# Step P3: Create executable output directory:
outdir = os.path.join(Ccodesrootdir, "output")
cmd.mkdir(outdir)

# Step P4: Enable/disable SIMD. If enabled, code should run ~2x faster on most CPUs.
enable_SIMD = True

# Step P5: Enable reference metric precomputation.
enable_rfm_precompute = False
if enable_rfm_precompute:
    par.set_parval_from_str("reference_metric::rfm_precompute_to_Cfunctions_and_NRPy_basic_defines", "True")
else:
    par.set_parval_from_str("reference_metric::rfm_precompute_to_Cfunctions_and_NRPy_basic_defines", "False")

# if enable_SIMD and not enable_rfm_precompute:
#     print("ERROR: SIMD does not currently handle transcendental functions,\n")
#     print("       like those found in rfmstruct (rfm_precompute).\n")
#     print("       Therefore, enable_SIMD==True and enable_rfm_precompute==False\n")
#     print("       is not supported.\n")
#     sys.exit(1)

# Step P6: Enable "FD functions". In other words, all finite-difference stencils
#         will be output as inlined static functions. This is essential for
#         compiling highly complex FD kernels with using certain versions of GCC;
#         GCC 10-ish will choke on BSSN FD kernels at high FD order, sometimes
#         taking *hours* to compile. Unaffected GCC versions compile these kernels
#         in seconds. FD functions do not slow the code performance, but do add
#         another header file to the C source tree.
# With gcc 7.5.0, enable_FD_functions=True decreases performance by 10%
enable_FD_functions = False

# Step 1: Set some core parameters, including CoordSystem, boundary condition,
#                                             MoL, timestepping algorithm, FD order,
#                                             floating point precision, and CFL factor:

# Step 1.a: Set the coordinate system for the numerical grid
# Choices are: Spherical, SinhSpherical, SinhSphericalv2, Cylindrical, SinhCylindrical,
#              SymTP, SinhSymTP
CoordSystem     = "Cartesian"
par.set_parval_from_str("reference_metric::CoordSystem", CoordSystem)
rfm.reference_metric()

# Step 1.c: Set defaults for Coordinate system parameters.
#           These are perhaps the most commonly adjusted parameters,
#           so we enable modifications at this high level.

# domain_size sets the default value for:
#   * Spherical's params.RMAX
#   * SinhSpherical*'s params.AMAX
#   * Cartesians*'s -params.{x,y,z}min & .{x,y,z}max
#   * Cylindrical's -params.ZMIN & .{Z,RHO}MAX
#   * SinhCylindrical's params.AMPL{RHO,Z}
#   * *SymTP's params.AMAX
domain_size     = 0.65 # Needed for all coordinate systems.

# sinh_width sets the default value for:
#   * SinhSpherical's params.SINHW
#   * SinhCylindrical's params.SINHW{RHO,Z}
#   * SinhSymTP's params.SINHWAA
sinh_width      = 0.4 # If Sinh* coordinates chosen

# sinhv2_const_dr sets the default value for:
#   * SinhSphericalv2's params.const_dr
#   * SinhCylindricalv2's params.const_d{rho,z}
sinhv2_const_dr = 0.05# If Sinh*v2 coordinates chosen

# SymTP_bScale sets the default value for:
#   * SinhSymTP's params.bScale
SymTP_bScale    = 1.0 # If SymTP chosen

# Step 1.d: Set the order of spatial and temporal derivatives;
#           the core data type, and the CFL factor.
# RK_method choices include: Euler, "RK2 Heun", "RK2 MP", "RK2 Ralston", RK3, "RK3 Heun", "RK3 Ralston",
#              SSPRK3, RK4, DP5, DP5alt, CK5, DP6, L6, DP8
RK_method = "RK4"
FD_order  = 6        # Finite difference order: even numbers only, starting with 2. 12 is generally unstable
REAL      = "double" # Best to use double here.
CFL_FACTOR= 0.5
outer_bc_type = "RADIATION_OUTER_BCS"  # can be EXTRAPOLATION_OUTER_BCS or RADIATION_OUTER_BCS
radiation_BC_FD_order = 2

In [2]:
# Step 5: Set the finite differencing order to FD_order (set above).
par.set_parval_from_str("finite_difference::FD_CENTDERIVS_ORDER", FD_order)

# Directory for reference_metric precomputation header files:
rfm_precompute_Ccode_outdir = os.path.join(Ccodesrootdir, "rfm_files/")
if enable_rfm_precompute:
    cmd.mkdir(os.path.join(Ccodesrootdir, "rfm_files/"))
    par.set_parval_from_str("reference_metric::rfm_precompute_Ccode_outdir", rfm_precompute_Ccode_outdir)

# Step 6: Copy SIMD/SIMD_intrinsics.h to $Ccodesrootdir/SIMD/SIMD_intrinsics.h
if enable_SIMD:
    cmd.mkdir(os.path.join(Ccodesrootdir,"SIMD"))
    shutil.copy(os.path.join("SIMD/")+"SIMD_intrinsics.h",os.path.join(Ccodesrootdir,"SIMD/"))

# Step 7: Set finite_difference::enable_FD_functions appropriately. Defaults to False
if enable_FD_functions:
    par.set_parval_from_str("finite_difference::enable_FD_functions", enable_FD_functions)

# Step 8: If enable_SIMD, then copy SIMD/SIMD_intrinsics.h to $Ccodesrootdir/SIMD/SIMD_intrinsics.h
cmd.mkdir(os.path.join(Ccodesrootdir,"SIMD"))
if enable_SIMD:
    shutil.copy(os.path.join("SIMD", "SIMD_intrinsics.h"), os.path.join(Ccodesrootdir, "SIMD"))


In [3]:
alpha = gri.register_gridfunctions("AUXEVOL","alpha")
betaU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","betaU", DIM=3)
gammaDD = ixp.register_gridfunctions_for_single_rank2("AUXEVOL", "gammaDD", "sym01")
KDD = ixp.register_gridfunctions_for_single_rank2("AUXEVOL", "KDD", "sym01")

phi_bssn = gri.register_gridfunctions("AUXEVOL","phi_bssn")
psi_bssn = gri.register_gridfunctions("AUXEVOL","psi_bssn")
gamma_tilde4UU =  ixp.register_gridfunctions_for_single_rank2("AUXEVOL", "gammaDD",
                                                          "sym01", DIM=4)

alpha_face = gri.register_gridfunctions("AUXEVOL","alpha_face")
beta_faceU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","beta_faceU", DIM=3)
gamma_faceDD = ixp.register_gridfunctions_for_single_rank2("AUXEVOL", "gamma_faceDD", "sym01")

vU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vU",DIM=3)
u4Ut = gri.register_gridfunctions("AUXEVOL","u4Ut")

vrU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vrU",DIM=3)
vrrU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vrrU",DIM=3)
vlrU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vlrU",DIM=3)

vlU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vlU",DIM=3)
vrlU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vlrU",DIM=3)
vllU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","vllU",DIM=3)

# u4rUt = gri.register_gridfunctions("AUXEVOL","u4rUt")
# u4lUt = gri.register_gridfunctions("AUXEVOL","u4lUt")

B_staggerU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","B_staggerU",DIM=3)
B_staggerrU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","B_staggerrU",DIM=3)
B_staggerlU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","B_staggerlU",DIM=3)
B_centerU = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","B_centerU",DIM=3)

StildeD = ixp.register_gridfunctions_for_single_rank1("EVOL","StildeD")
rho_star = gri.register_gridfunctions("EVOL","rho_star")
tau_tilde = gri.register_gridfunctions("EVOL","tau_tilde")

AD = ixp.register_gridfunctions_for_single_rank1("EVOL","AD")
phitilde = gri.register_gridfunctions("EVOL","phitilde")

Stilde_flux_HLLD = ixp.register_gridfunctions_for_single_rank1("AUXEVOL","Stilde_flux_HLLD")
rho_star_flux_HLL = gri.register_gridfunctions("AUXEVOL","rho_star_flux_HLL")
tau_tilde_flux_HLL = gri.register_gridfunctions("AUXEVOL","tau_tilde_flux_HLL")

rhob = gri.register_gridfunctions("AUXEVOL","rhob")
rhob_r = gri.register_gridfunctions("AUXEVOL","rhob_r")
rhob_l = gri.register_gridfunctions("AUXEVOL","rhob_l")
P = gri.register_gridfunctions("AUXEVOL","P")
P_r = gri.register_gridfunctions("AUXEVOL","P_r")
P_l = gri.register_gridfunctions("AUXEVOL","P_l")


cmin_x = gri.register_gridfunctions("AUXEVOL","cmin_x")
cmax_x = gri.register_gridfunctions("AUXEVOL","cmax_x")
cmin_y = gri.register_gridfunctions("AUXEVOL","cmin_y")
cmax_y = gri.register_gridfunctions("AUXEVOL","cmax_y")
cmin_z = gri.register_gridfunctions("AUXEVOL","cmin_z")
cmax_z = gri.register_gridfunctions("AUXEVOL","cmax_z")

In [4]:
import ShockTests.ShockTests_1D as st_1d
import ShockTests.ShockTests_2D as st_2d

ID_opts = ["Balsara1", "Balsara2", "Balsara3", "Balsara4", "Balsara5",
           "Hydro1", "Hydro2",
          "Cylindrical_Explosion", "Loop_Advection", "Magnetic_Rotor"]

def add_to_Cfunction_dict_ID_options():
    par.set_parval_from_str("reference_metric::CoordSystem", "Cartesian")
    rfm.reference_metric()

    x = rfm.xx[0]
    y = rfm.xx[1]
    z = rfm.xx[2]

    r = rfm.xxSph[0]

    for initial_data in ID_opts:
        if initial_data=="Balsara1":
            rho, press, vU, BU = st_1d.balsara1(x, bound=0.05)
            desc = "Generate Balsara1 1D initial data"
        elif initial_data=="Balsara2":
            rho, press, vU, BU = st_1d.balsara2(x, bound=0.05)
            desc = "Generate Balsara2 1D initial data"
        elif initial_data=="Balsara3":
            rho, press, vU, BU = st_1d.balsara3(x, bound=0.05)
            desc = "Generate Balsara3 1D initial data"
        elif initial_data=="Balsara4":
            rho, press, vU, BU = st_1d.balsara4(x, bound=0.05)
            desc = "Generate Balsara4 1D initial data"
        elif initial_data=="Balsara5":
            rho, press, vU, BU = st_1d.balsara5(x, bound=0.05)
            desc = "Generate Balsara5 1D initial data"
        elif initial_data=="Hydro1":
            rho, press, vU, BU = st_1d.hydro1(x, bound=0.05)
            desc = "Generate mildy relativistic 1D hydro initial data"
        elif initial_data=="Hydro2":
            rho, press, vU, BU = st_1d.hydro2(x, bound=0.05)
            desc = "Generate hydro blast wave 1D initial data"
        elif initial_data=="Cylindrical_Explosion":
            rho, press, vU, BU = st_2d.cylindrical_explosion(r)
            desc = "Generate 2D Cylindrical Explosion initial data"
        elif initial_data=="Magnetic_Rotor":
            rho, press, vU, BU = st_2d.magnetic_rotor(r, cartx=x, carty=y, cartz=z)
            desc = "Generate 2D Magnetic Rotor initial data"
        elif initial_data=="Loop_Advection":
            rho, press, vU, AD = st_2d.loop_advection(r)
            desc = "Generate 2D Loop Advection initial data"

        if initial_data!="Loop_Advection":
            AD = st_1d.BtoA_piecewise_constant_Cart_Flat(x, y, z, BU)

        name = initial_data

        gammaDD = ixp.zerorank2()
        gammaDD[0][0] = sp.sympify(1.0)
        gammaDD[1][1] = sp.sympify(1.0)
        gammaDD[2][2] = sp.sympify(1.0)

        betaU = ixp.zerorank1()

        alpha = sp.sympify(1)

        KDD = ixp.zerorank2()

        values_to_print = [
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","vU0"),rhs=vU[0]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","vU1"),rhs=vU[1]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","vU2"),rhs=vU[2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","rhob"),rhs=rho),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","P"),rhs=press),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","gammaDD00"),rhs=gammaDD[0][0]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","gammaDD01"),rhs=gammaDD[0][1]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","gammaDD02"),rhs=gammaDD[0][2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","gammaDD11"),rhs=gammaDD[1][1]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","gammaDD12"),rhs=gammaDD[1][2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","gammaDD22"),rhs=gammaDD[2][2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","KDD00"),rhs=KDD[0][0]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","KDD01"),rhs=KDD[0][1]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","KDD02"),rhs=KDD[0][2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","KDD11"),rhs=KDD[1][1]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","KDD12"),rhs=KDD[1][2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","KDD22"),rhs=KDD[2][2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","betaU0"),rhs=betaU[0]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","betaU1"),rhs=betaU[1]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","betaU2"),rhs=betaU[2]),
                       lhrh(lhs=gri.gfaccess("auxevol_gfs","alpha"),rhs=alpha)
                      ]

        includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
        c_type = "void"
        params   ="const paramstruct *restrict params,REAL *restrict xx[3], REAL *restrict evol_gfs, REAL *restrict auxevol_gfs"
        body = fin.FD_outputC("returnstring",values_to_print,params="outCverbose=False")
        add_to_Cfunction_dict(
            includes=includes,
            desc=desc,
            c_type=c_type, name=name, params=params,
            body=body,
            rel_path_to_Cparams=os.path.join("."), loopopts="AllPoints,Read_xxs")


In [5]:
# say what data its using at run time

def add_to_Cfunction_dict_initial_data_driver():
    inital_data_body = r"""

    const char *option1 = "b1";
    const char *option2 = "b2";
    const char *option3 = "b3";
    const char *option4 = "b4";
    const char *option5 = "b5";
    const char *option6 = "h1";
    const char *option7 = "h2";
    const char *option8 = "ce";
    //const char *option9 = "mr";
    const char *option9 = "la";
    const char *option10 = "read";

    if (strcmp(initial_data_option, option1) == 0) {
        Balsara1(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option2) == 0) {
        Balsara2(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option3) == 0) {
        Balsara3(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option4) == 0) {
        Balsara4(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option5) == 0) {
        Balsara5(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option6) == 0) {
        Hydro1(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option7) == 0) {
        Hydro2(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option8) == 0) {
        Cylindrical_Explosion(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option9) == 0) {
        Magnetic_Rotor(params, xx, evol_gfs, auxevol_gfs);
    }

    else if (strcmp(initial_data_option, option9) == 0) {
        Loop_Advection(params, xx, evol_gfs, auxevol_gfs);
    }
    else {
        printf("ERROR: Invalid choice of initial data.\n");
        exit(1);
    }
"""
    name = "initial_data"
    desc = "Main initial data function."
    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    add_to_Cfunction_dict(
        name     = name,
        desc     = desc,
        params   ="const char *initial_data_option, const paramstruct *restrict params, REAL *restrict xx[3], REAL *restrict evol_gfs,REAL *restrict auxevol_gfs",
        body     = inital_data_body,
        includes = includes,
        rel_path_to_Cparams=os.path.join("."))


In [15]:
def add_to_Cfunction_dict__interpolate_metric_to_face():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "interpolate_metric_to_face"
    params = r"""
const int i, const int j, const int k,
const int flux_dir,
const REAL *restrict lapse,
const REAL *restrict betax,
const REAL *restrict betay,
const REAL *restrict betaz,
const REAL *restrict gxx,
const REAL *restrict gxy,
const REAL *restrict gxz,
const REAL *restrict gyy,
const REAL *restrict gyz,
const REAL *restrict gzz,
metric_quantities *restrict metric"""

    body = r"""
  const int xdir = (flux_dir == 0);
  const int ydir = (flux_dir == 1);
  const int zdir = (flux_dir == 2);

  const int indm2  = IDX3S(i-2*xdir, j-2*ydir, k-2*zdir);
  const int indm1  = IDX3S(i-xdir,   j-ydir,   k-zdir);
  const int index  = IDX3S(i,        j ,       k);
  const int indp1  = IDX3S(i+xdir,   j+ydir,   k+zdir);

  const REAL face_lapse = COMPUTE_FCVAL(lapse[indm2], lapse[indm1], lapse[index], lapse[indp1]);
  const REAL face_betax = COMPUTE_FCVAL(betax[indm2], betax[indm1], betax[index], betax[indp1]);
  const REAL face_betay = COMPUTE_FCVAL(betay[indm2], betay[indm1], betay[index], betay[indp1]);
  const REAL face_betaz = COMPUTE_FCVAL(betaz[indm2], betaz[indm1], betaz[index], betaz[indp1]);

  const REAL face_gxx = COMPUTE_FCVAL(gxx[indm2], gxx[indm1], gxx[index], gxx[indp1]);
  const REAL face_gxy = COMPUTE_FCVAL(gxy[indm2], gxy[indm1], gxy[index], gxy[indp1]);
  const REAL face_gxz = COMPUTE_FCVAL(gxz[indm2], gxz[indm1], gxz[index], gxz[indp1]);
  const REAL face_gyy = COMPUTE_FCVAL(gyy[indm2], gyy[indm1], gyy[index], gyy[indp1]);
  const REAL face_gyz = COMPUTE_FCVAL(gyz[indm2], gyz[indm1], gyz[index], gyz[indp1]);
  const REAL face_gzz = COMPUTE_FCVAL(gzz[indm2], gzz[indm1], gzz[index], gzz[indp1]);

  ghl_initialize_metric(
        face_lapse,
        face_betax, face_betay, face_betaz,
        face_gxx, face_gxy, face_gxz,
        face_gyy, face_gyz, face_gzz,
        metric);
"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [16]:
def add_to_Cfunction_dict__compute_metric_derivs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    desc = "compute metric derivatives"
    name = "compute_metric_derivs"
    params = r"""
const int i, const int j, const int k,
const int flux_dir,
const REAL *restrict lapse,
const REAL *restrict betax,
const REAL *restrict betay,
const REAL *restrict betaz,
const REAL *restrict gxx,
const REAL *restrict gxy,
const REAL *restrict gxz,
const REAL *restrict gyy,
const REAL *restrict gyz,
const REAL *restrict gzz,
metric_quantities *restrict metric_derivs"""

    body = r"""
  const int xdir = (flux_dir == 0);
  const int ydir = (flux_dir == 1);
  const int zdir = (flux_dir == 2);

  const int indm2  = IDX3S(i-2*xdir, j-2*ydir, k-2*zdir);
  const int indm1  = IDX3S(i-xdir,   j-ydir,   k-zdir);
  const int indp1  = IDX3S(i+xdir,   j+ydir,   k+zdir);
  const int indp2  = IDX3S(i+2*xdir, j+2*ydir, k+2*zdir);

  const REAL d_lapse = dxi*COMPUTE_DERIV(lapse[indm2], lapse[indm1], lapse[indp1], lapse[indp2]);
  const REAL d_betax = dxi*COMPUTE_DERIV(betax[indm2], betax[indm1], betax[indp1], betax[indp2]);
  const REAL d_betay = dxi*COMPUTE_DERIV(betay[indm2], betay[indm1], betay[indp1], betay[indp2]);
  const REAL d_betaz = dxi*COMPUTE_DERIV(betaz[indm2], betaz[indm1], betaz[indp1], betaz[indp2]);

  const REAL d_gxx = dxi*COMPUTE_DERIV(gxx[indm2], gxx[indm1], gxx[indp1], gxx[indp2]);
  const REAL d_gxy = dxi*COMPUTE_DERIV(gxy[indm2], gxy[indm1], gxy[indp1], gxy[indp2]);
  const REAL d_gxz = dxi*COMPUTE_DERIV(gxz[indm2], gxz[indm1], gxz[indp1], gxz[indp2]);
  const REAL d_gyy = dxi*COMPUTE_DERIV(gyy[indm2], gyy[indm1], gyy[indp1], gyy[indp2]);
  const REAL d_gyz = dxi*COMPUTE_DERIV(gyz[indm2], gyz[indm1], gyz[indp1], gyz[indp2]);
  const REAL d_gzz = dxi*COMPUTE_DERIV(gzz[indm2], gzz[indm1], gzz[indp1], gzz[indp2]);

  ghl_initialize_metric(
        d_lapse,
        d_betax, d_betay, d_betaz,
        d_gxx, d_gxy, d_gxz,
        d_gyy, d_gyz, d_gzz,
        metric_derivs);
"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [17]:
def add_to_Cfunction_dict__calculate_MHD_dirn_rhs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "calculate_MHD_dirn_rhs"
    params = r"""
const paramstruct *restrict params,
const eos_parameters *restrict ghl_eos,
const int flux_dir,
const REAL **B_center,
const REAL *restrict B_stagger,
REAL **vel_r,
REAL **vel_l,
REAL *restrict cmin,
REAL *restrict cmax,
const REAL *restrict in_gfs,
REAL *auxevol_gfs,
REAL *restrict rhs_gfs"""

    prefunc = r"""
REAL eos_Gamma_eff(const eos_parameters *restrict ghl_eos, const REAL rho_in, const REAL press_in) {
  REAL K, Gamma;
  ghl_hybrid_get_K_and_Gamma(ghl_eos, rho_in, &K, &Gamma);
  const REAL P_cold = K*pow(rho_in, Gamma);
  return eos->Gamma_th + (Gamma - eos->Gamma_th)*P_cold/press_in;
}

void reconstruct_v_edges(
      co ghl_eos,nst int flux_dir,
      const eos_parameters *restrict ghl_eos,
      const int imin,
      const int imax,
      const int jmin,
      const int jmax,
      const int kmin,
      const int kmax,
      REAL **vel_r,
      REAL **vel_l,
      REAL *auxevol_gfs) {

  const int xdir = (flux_dir == 0);
  const int ydir = (flux_dir == 1);
  const int zdir = (flux_dir == 2);

#pragma omp parallel for
  for(int k=kmin; k<kmax; k++) {
    for(int j=jmin; j<jmax; j++) {
      for(int i=imin; i<imax; i++) {
        const int index = IDX3S(i, j, k);

        REAL press_stencil[6], v_flux_dir[6];
        REAL var_data[3][6], vars_r[3], vars_l[3];

        for(int ind=0; ind<6; ind++) {
          // Stencil from -3 to +2 reconstructs to e.g. i-1/2
          const int stencil = IDX3S(i+xdir*(ind-3), j+ydir*(ind-3), k+zdir*(ind-3));
          v_flux_dir[ind] = v_flux[stencil]; // Could be smaller; doesn't use full stencil
          press_stencil[ind] = auxevol_gfs[IDX4ptS(PGF, stencil)];
          var_data[0][ind] = auxevol_gfs[IDX4ptS(VU0GF, stencil)];
          var_data[1][ind] = auxevol_gfs[IDX4ptS(VU1GF, stencil)];
          var_data[2][ind] = auxevol_gfs[IDX4ptS(VU2GF, stencil)];
        }

        // Compute Gamma
        const REAL Gamma = eos_Gamma_eff(ghl_eos,
                                         auxevol_gfs[IDX4ptS(RHOBGF, index)],
                                         auxevol_gfs[IDX4ptS(PGF, index)]);

        ghl_ppm_no_rho_P(
              press_stencil, var_data,
              3, v_flux_dir, Gamma,
              vars_r, vars_l);

        vel_r[0][index] = vars_r[0];
        vel_r[1][index] = vars_r[1];
        vel_r[2][index] = vars_r[2];

        vel_l[0][index] = vars_l[0];
        vel_l[1][index] = vars_l[1];
        vel_l[2][index] = vars_l[2];
      }
    }
  }
}

"""

    body = r"""

// We could reduce computational cost by introducing grid functions for the
// metric face interpolations (used in both loops

 const REAL dxi = 1.0/dX[flux_dir];
  const REAL poison = 0.0/0.0;

  // Function pointer to allow for loop over fluxes and sources
  void (*calculate_characteristic_speed)(const ghl_primitive_quantities *restrict prims_r,
                                         const ghl_primitive_quantities *restrict prims_l,
                                         const ghl_eos_parameters *restrict eos,
                                         const ghl_metric_quantities *restrict ADM_metric_face,
                                         REAL *cmin, REAL *cmax);

  void (*calculate_HLLE_fluxes)(const ghl_primitive_quantities *restrict prims_r,
                                const ghl_primitive_quantities *restrict prims_l,
                                const ghl_eos_parameters *restrict eos,
                                const ghl_metric_quantities *restrict ADM_metric_face,
                                const REAL cmin,
                                const REAL cmax,
                                ghl_conservative_quantities *restrict cons_fluxes);

  void (*calculate_source_terms)(const ghl_primitive_quantities *restrict prims,
                                 const ghl_eos_parameters *restrict eos,
                                 const ghl_metric_quantities *restrict ADM_metric,
                                 const ghl_metric_quantities *restrict metric_derivs,
                                 ghl_conservative_quantities *restrict cons_sources);

  const int xdir = (flux_dir == 0);
  const int ydir = (flux_dir == 1);
  const int zdir = (flux_dir == 2);

  const REAL *v_flux;
  int B_recon[3];
  // Set function pointer to specific function for a given direction
  switch(flux_dir) {
    case 0:
      v_flux = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VU0GF;
      B_recon[0] = 0;
      B_recon[1] = 1;
      B_recon[2] = 2;
      calculate_characteristic_speed = &ghl_calculate_characteristic_speed_dirn0;
      calculate_HLLE_fluxes = &ghl_calculate_HLLE_fluxes_dirn0;
      calculate_source_terms = &ghl_calculate_source_terms_dirn0;
      break;
    case 1:
      v_flux = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VU1GF;
      B_recon[0] = 1;
      B_recon[1] = 2;
      B_recon[2] = 0;
      calculate_characteristic_speed = &ghl_calculate_characteristic_speed_dirn1;
      calculate_HLLE_fluxes = &ghl_calculate_HLLE_fluxes_dirn1;
      calculate_source_terms = &ghl_calculate_source_terms_dirn1;
      break;
    case 2:
      v_flux = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VU2GF;
      B_recon[0] = 2;
      B_recon[1] = 0;
      B_recon[2] = 1;
      calculate_characteristic_speed = &ghl_calculate_characteristic_speed_dirn2;
      calculate_HLLE_fluxes = &ghl_calculate_HLLE_fluxes_dirn2;
      calculate_source_terms = &ghl_calculate_source_terms_dirn2;
      break;
    default:
      printf("Warning: invalid flux_dir value (not 0, 1, or 2) has been passed to calculate_MHD_rhs.");
  }


  const int imin = NGHOSTS;
  const int jmin = NGHOSTS;
  const int kmin = NGHOSTS;
  const int imax = Nxx_plus_2NGHOSTS0 - NGHOSTS;
  const int jmax = Nxx_plus_2NGHOSTS1 - NGHOSTS;
  const int kmax = Nxx_plus_2NGHOSTS2 - NGHOSTS;

#pragma omp parallel for
  for(int k=0; k<Nxx_plus_2NGHOSTS2 - NGHOSTS; k++) {
    for(int j=0; j<Nxx_plus_2NGHOSTS1 - NGHOSTS; j++) {
      for(int i=0; i<Nxx_plus_2NGHOSTS0 - NGHOSTS; i++) {
        const int index = IDX3S(i, j, k);
        vel_r[0][index] = poison;
        vel_r[1][index] = poison;
        vel_r[2][index] = poison;

        vel_l[0][index] = poison;
        vel_l[1][index] = poison;
        vel_l[2][index] = poison;
      }
    }
  }
  // This loop includes 1 ghostzone because the RHS calculation for e.g. the x direction
  // requires (i,j,k) and (i+1,j,k); if cmin/max weren't also needed for A_i, we could
  // technically have the loop only go 1 extra point in the flux_dir direction
#pragma omp parallel for
  for(int k=kmin; k<kmax+1; k++) {
    for(int j=jmin; j<jmax+1; j++) {
      for(int i=imin; i<imax+1; i++) {
        const int indm1 = IDX3S(i-xdir, j-ydir, k-zdir); /* indexim1=0 when i=0 */
        const int index = IDX3S(i, j, k);

        REAL rho_stencil[6], press_stencil[6], v_flux_dir[6];
        REAL rhor, rhol, pressr, pressl, B_r[3], B_l[3];
        REAL var_data[5][6], vars_r[5], vars_l[5];

        for(int ind=0; ind<6; ind++) {
          // Stencil from -3 to +2 reconstructs to e.g. i-1/2
          const int stencil = IDX3S(i+xdir*(ind-3), j+ydir*(ind-3), k+zdir*(ind-3));
          v_flux_dir[ind] = v_flux[stencil]; // Could be smaller; doesn't use full stencil
          rho_stencil[ind] = auxevol_gfs[IDX4ptS(RHOBGF, stencil)];
          press_stencil[ind] = auxevol_gfs[IDX4ptS(PGF, stencil)];
          var_data[0][ind] = auxevol_gfs[IDX4ptS(VU0GF, stencil)];
          var_data[1][ind] = auxevol_gfs[IDX4ptS(VU1GF, stencil)];
          var_data[2][ind] = auxevol_gfs[IDX4ptS(VU2GF, stencil)];
          var_data[3][ind] = B_center[B_recon[1]][stencil];
          var_data[4][ind] = B_center[B_recon[2]][stencil];
        }

        // Compute Gamma
        const REAL Gamma = eos_Gamma_eff(ghl_eos,
                                         auxevol_gfs[IDX4ptS(RHOBGF, index)],
                                         auxevol_gfs[IDX4ptS(PGF, index)]);

        ghl_ppm(
              rho_stencil, press_stencil, var_data,
              5, v_flux_dir, Gamma,
              &rhor, &rhol, &pressr, &pressl, vars_r, vars_l);

        vel_r[0][index] = vars_r[0];
        vel_r[1][index] = vars_r[1];
        vel_r[2][index] = vars_r[2];

        vel_l[0][index] = vars_l[0];
        vel_l[1][index] = vars_l[1];
        vel_l[2][index] = vars_l[2];

        B_r[B_recon[1]] = vars_r[3];
        B_r[B_recon[2]] = vars_r[4];

        B_l[B_recon[1]] = vars_l[3];
        B_l[B_recon[2]] = vars_l[4];

        ghl_metric_quantities ADM_metric_face;
        interpolate_metric_to_face(
              i, j, k,
              flux_dir,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*ALPHAGF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*BETAU0GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*BETAU1GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*BETAU2GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*GAMMADD00GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*GAMMADD01GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*GAMMADD02GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*GAMMADD11GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*GAMMADD12GF,
              auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*GAMMADD22GF,
              &ADM_metric_face);

        // B_stagger is densitized, but B_center is not.
        B_r[B_recon[0]] = B_stagger[indm1]/ADM_metric_face.sqrt_detgamma;
        B_l[B_recon[0]] = B_stagger[indm1]/ADM_metric_face.sqrt_detgamma;

        ghl_primitive_quantities prims_r, prims_l;
        ghl_initialize_primitives(
              rhor, pressr, poison,
              vel_r[0][index], vel_r[1][index], vel_r[2][index],
              B_r[0], B_r[1], B_r[2],
              poison, poison, poison, // entropy, Y_e, temp
              &prims_r);

        ghl_initialize_primitives(
              rhol, pressl, poison,
              vel_l[0][index], vel_l[1][index], vel_l[2][index],
              B_l[0], B_l[1], B_l[2],
              poison, poison, poison, // entropy, Y_e, temp
              &prims_l);

        int speed_limited = 0;
        ghl_limit_v_and_compute_u0(
              ghl_eos, &ADM_metric_face, &prims_r, &speed_limited);
        ghl_limit_v_and_compute_u0(
              ghl_eos, &ADM_metric_face, &prims_l, &speed_limited);

        ghl_conservative_quantities cons_fluxes;
        calculate_characteristic_speed(&prims_r, &prims_l, ghl_eos, &ADM_metric_face, &cmin[index], &cmax[index]);
        calculate_HLLE_fluxes(&prims_r, &prims_l, ghl_eos, &ADM_metric_face, cmin[index], cmax[index], &cons_fluxes);

        REAL dum1, dum2;
        ghl_return_conservatives(&cons_fluxes,
                           &auxevol_gfs[IDX4ptS(RHO_STAR_FLUX_HLLGF, index)],
                           &auxevol_gfs[IDX4ptS(TAU_TILDE_FLUX_HLLGF, index)],
                           &auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD0GF, index)],
                           &auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD1GF, index)],
                           &auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD2GF, index)],
                           &dum1, &dum2);
      }
    }
  }

   /*
     The following loops fill in the outer ghost zones for the
     reconstructed velocities. To compute the flux terms for A_RHS, we
     need to reconstruct the velocities a second time. This means we
     will need to do another reconstruction stencil in the interior.
     However, the previous loop only provides the minimum loop size.
     We could needlessly compute lots of extra math in the previous
     loop to simplify the code, but instead we loop over the extra
     ghost zone points here and just reconstruct the velocities.

     Example: consider the x direction. If we reconstruct in x first,
     then we may want to reconstruct in y or z. Then, the full grid
     we want is
        i: loop over interior + 1 one upper ghost zone (due to different centerings)
        j: loop over all points
        k: loop over all points
     We can visualize this with a box

              ------------
              |          |
              | -------- |
              | |xxxxxx| |
              | |xxxxxx| |
              | |xxxxxx| |
           ^  | -------- |
           |  |          |
           k  ------------
           j-->

     where the 'x' positions have already been filled.
     We can do several loops to fill the remainder:
     1) loop k @ (0, kmin), j @ (0,lsh[1]), i @ (imin,imax+1)
        loop shown with w
     2) loop k @ (kmax+1, lsh[2]), j @ (0,lsh[1]), i @ (imin,imax+1)
        loop shown with y
     3) loop k @ (kmin, kmax+1), j @ (0,jmin), i @ (imin,imax+1)
        loop shown with z
     4) loop k @ (kmin, kmax+1), j @ (jmax+1,lsh[1]), i @ (imin,imax+1)
        loop shown with q
              ------------
              |yyyyyyyyyy|
              |z--------q|
              |z|xxxxxx|q|
              |z|xxxxxx|q|
              |z|xxxxxx|q|
              |z--------q|
              |wwwwwwwwww|
              ------------
  */
  if(xdir) {
    reconstruct_v_edges(flux_dir, ghl_eos, // w zone
          imin, imax+1,
          0, Nxx_plus_2NGHOSTS1 - NGHOSTS,
          0, kmin,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos, // y zone
          imin, imax+1,
          0, Nxx_plus_2NGHOSTS1 - NGHOSTS,
          kmax+1, Nxx_plus_2NGHOSTS2 - NGHOSTS,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos, // z zone
          imin, imax+1,
          0, jmin,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos,  // q zone
          imin, imax+1,
          jmax+1, Nxx_plus_2NGHOSTS1 - NGHOSTS,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);
  } else if(ydir) {
    reconstruct_v_edges(flux_dir, ghl_eos, // w zone
          0, Nxx_plus_2NGHOSTS0 - NGHOSTS,
          jmin, jmax+1,
          0, kmin,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos, // y zone
          0, Nxx_plus_2NGHOSTS0 - NGHOSTS,
          jmin, jmax+1,
          kmax+1, Nxx_plus_2NGHOSTS2 - NGHOSTS,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos, // z zone
          0, imin,
          jmin, jmax+1,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos,  // q zone
          imax+1, Nxx_plus_2NGHOSTS0 - NGHOSTS,
          jmin, jmax+1,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);
  } else if(zdir) {
    reconstruct_v_edges(flux_dir, ghl_eos, // w zone
          0, Nxx_plus_2NGHOSTS0 - NGHOSTS,
          0, jmin,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos, // y zone
          0, Nxx_plus_2NGHOSTS0 - NGHOSTS,
          jmax+1, Nxx_plus_2NGHOSTS1 - NGHOSTS,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos, // z zone
          0, imin,
          jmin, jmax+1,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);

    reconstruct_v_edges(flux_dir, ghl_eos,  // q zone
          imax+1, Nxx_plus_2NGHOSTS0 - NGHOSTS,
          jmin, jmax+1,
          kmin, kmax+1,
          v_flux, vel_r, vel_l);
  }

#pragma omp parallel for
  for(int k=kmin; k<kmax; k++) {
    for(int j=jmin; j<jmax; j++) {
      for(int i=imin; i<imax; i++) {

          const int xdir = (flux_dir == 0);
          const int ydir = (flux_dir == 1);
          const int zdir = (flux_dir == 2);

          const int index = IDX3S(i0, i1, i2);
          const int indexp1 = IDX3S(i0+xdir, i1+ydir, i2+zdir);

          rhs_gfs[IDX4ptS(RHO_STARGF, index)]  += invdx*(auxevol_gfs[IDX4ptS(RHO_STAR_FLUX_HLLGF, index)] -
                                                         auxevol_gfs[IDX4ptS(RHO_STAR_FLUX_HLLGF, indexp1)]);

          rhs_gfs[IDX4ptS(TAU_TILDEGF, index)] += invdx*(auxevol_gfs[IDX4ptS(TAU_TILDE_FLUX_HLLGF, index)] -
                                                         auxevol_gfs[IDX4ptS(TAU_TILDE_FLUX_HLLGF, indexp1)]);

          rhs_gfs[IDX4ptS(STILDED0GF, index)]  += invdx*(auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD0GF, index)] -
                                                         auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD0GF, indexp1)]);

          rhs_gfs[IDX4ptS(STILDED1GF, index)]  += invdx*(auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD1GF, index)] -
                                                         auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD1GF, indexp1)]);

          rhs_gfs[IDX4ptS(STILDED2GF, index)]  += invdx*(auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD2GF, index)] -
                                                         auxevol_gfs[IDX4ptS(STILDE_FLUX_HLLD2GF, indexp1)]);

        metric_quantities ADM_metric;
        ghl_initialize_metric(auxevol_gfs[IDX4ptS(ALPHAGF, index)],
                              auxevol_gfs[IDX4ptS(BETAU0GF, index)],
                              auxevol_gfs[IDX4ptS(BETAU1GF, index)],
                              auxevol_gfs[IDX4ptS(BETAU2GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD00GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD01GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD02GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD11GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD12GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD22GF, index)],
                              &ADM_metric);

        primitive_quantities prims;
        ghl_initialize_primitives(
              auxevol_gfs[IDX4ptS(RHOBGF, index)],
              auxevol_gfs[IDX4ptS(PGF, index)],
              poison,
              auxevol_gfs[IDX4ptS(VU0GF, index)],
              auxevol_gfs[IDX4ptS(VU1GF, index)],
              auxevol_gfs[IDX4ptS(VU2GF, index)],
              B_center[0][index], B_center[1][index], B_center[2][index],
              poison, poison, poison, // entropy, Y_e, temp
              &prims);

        int speed_limited = 0;
        ghl_limit_v_and_compute_u0(
              ghl_eos, &ADM_metric, &prims, &speed_limited);

        metric_quantities ADM_metric_derivs;
        compute_metric_derivs(
              i, j, k,
              flux_dir, dxi,
              auxevol_gfs[IDX4ptS(ALPHAGF, index)],
              auxevol_gfs[IDX4ptS(BETAU0GF, index)],
              auxevol_gfs[IDX4ptS(BETAU1GF, index)],
              auxevol_gfs[IDX4ptS(BETAU2GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD00GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD01GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD02GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD11GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD12GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD22GF, index)],
              &ADM_metric_derivs);

        conservative_quantities cons_source;
        cons_source.tau = 0.0;
        cons_source.SD[0] = 0.0;
        cons_source.SD[1] = 0.0;
        cons_source.SD[2] = 0.0;

        calculate_source_terms(&prims, ghl_eos, &ADM_metric, &ADM_metric_derivs, &cons_source);
        rhs_gfs[IDX4ptS(TAU_TILDEGF, index)] += cons_source.tau;
        rhs_gfs[IDX4ptS(STILDED0GF, index)]  += cons_source.SD[0];
        rhs_gfs[IDX4ptS(STILDED1GF, index)]  += cons_source.SD[1];
        rhs_gfs[IDX4ptS(STILDED2GF, index)]  += cons_source.SD[2];
      }
    }
  }
"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        prefunc=prefunc,
        loopopts = "")

In [18]:
def add_to_Cfunction_dict__evaluate_MHD_rhs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]

    c_type = "void"
    name = "evaluate_MHD_rhs"
    params = r"""
const paramstruct *restrict params,
const eos_parameters *restrict eos,
const REAL *restrict in_gfs,
REAL *auxevol_gfs,
REAL *restrict rhs_gfs"""

    desc = r"""
/*********************************************
 * Evaluate RHS of GRMHD & induction equations
 * (vector potential prescription), using the
 * generalized Lorenz gauge condition for the
 * EM gauge.
 *
 * Based originally on the Illinois GRMHD code,
 * written by Matt Duez, Yuk Tung Liu, and Branson
 * Stephens (original version), and then developed
 * primarily by Zachariah Etienne, Yuk Tung Liu,
 * and Vasileios Paschalidis.
 *
 * Rewritten for public release in 2013
 *      by Zachariah B. Etienne
 *
 * References:
 * Original unigrid GRMHD evolution prescription:
 *    http://arxiv.org/abs/astro-ph/0503420
 * Vector potential formulation in full GR:
 *    http://arxiv.org/abs/1007.2848
 * Improved EM gauge conditions for AMR grids:
 *    http://arxiv.org/abs/1110.4633
 * Generalized Lorenz gauge prescription:
 *    http://arxiv.org/abs/1207.3354
 *
 * Note that the Generalized Lorenz gauge strength
 *  parameter has units of 1/M, just like the \eta
 *  parameter in the gamma-driving shift condition,
 *  so setting it too large will result in violation
 *  of the CFL condition.
 *
 * This version of PPM implements the standard
 * Colella & Woodward PPM, though modified as in GRHydro
 * to have 3 ghostzones instead of 4.
 *********************************************/
 """

    body = r"""
  const REAL dX[3] = { dxx0, dxx1, dxx2 };

  const int Nxx_plus_2NGHOSTS_TOT = Nxx_plus_2NGHOSTS0*Nxx_plus_2NGHOSTS1*Nxx_plus_2NGHOSTS2;

  // in_prims,out_prims_r, and out_prims_l are arrays of pointers to the actual gridfunctions.
  // Most pointers are passed explicitly. However, we need to programmatically choose gridfunctions
  // for the A_i reconstructions.
  const double *in_prims[MAXNUMVARS];
  double *out_prims_r[MAXNUMVARS];
  double *out_prims_l[MAXNUMVARS];

  in_prims   [BX_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERU0GF;
  out_prims_r[BX_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERRU0GF;
  out_prims_l[BX_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERLU0GF;

  in_prims   [BY_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERU1GF;
  out_prims_r[BY_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERRU1GF;
  out_prims_l[BY_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERLU1GF;

  in_prims   [BY_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERU2GF;
  out_prims_r[BY_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERRU2GF;
  out_prims_l[BY_STAGGER]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERLU2GF;

  in_prims   [VXR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU0GF;
  out_prims_r[VXR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRRU0GF;
  out_prims_l[VXR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRLU0GF;

  in_prims   [VYR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU1GF;
  out_prims_r[VYR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRRU1GF;
  out_prims_l[VYR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRLU1GF;

  in_prims   [VZR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU2GF;
  out_prims_r[VZR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRRU2GF;
  out_prims_l[VZR]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRLU2GF;

  in_prims   [VXL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU0GF;
  out_prims_r[VXL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLRU0GF;
  out_prims_l[VXL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLLU0GF;

  in_prims   [VYL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU1GF;
  out_prims_r[VYL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLRU1GF;
  out_prims_l[VYL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLLU1GF;

  in_prims   [VZL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU2GF;
  out_prims_r[VZL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLRU2GF;
  out_prims_l[VZL]=auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLLU2GF;

  const double *vel[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VU0GF,
                          auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VU1GF,
                          auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VU2GF};

  const double *B_center[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_CENTERU0GF,
                               auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_CENTERU1GF,
                               auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_CENTERU2GF};

  double *vel_r[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU0GF,
                      auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU1GF,
                      auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU2GF};

  double *vel_l[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU0GF,
                      auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU1GF,
                      auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU2GF};

  const double *B_stagger[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERU0GF,
                                auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERU1GF,
                                auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*B_STAGGERU2GF};

  double *cmin[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*CMIN_XGF,
                     auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*CMIN_YGF,
                     auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*CMIN_ZGF};

  double *cmax[3] = {auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*CMAX_XGF,
                     auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*CMAX_YGF,
                     auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*CMAX_ZGF};

  /*
     We first compute the x-direction RHS for the conservative hydro variables using PPM
     reconstruction from (i, j, k) to (i-1/2, j, k). This reconstructs rho_b, P, v^i, and
     B^i. B^x is not reconstructed, as this is given by the Bx_stagger grid function. This
     is all done to be able to compute
         \partial_x F = [ F(i+1/2,j,k) - F(i-1/2,j,k) ] / dx
     We also save the v^i reconstructions for the A_i RHS.
  */
  int flux_dir = 0;
  calculate_MHD_dirn_rhs(flux_dir, eos, dX, B_center, B_stagger[flux_dir],
                         vel_r, vel_l, cmin[flux_dir], cmax[flux_dir],
                         auxevol_gfs, in_gfs, rhs_gfs);

  /*
     Here we perform reconstructions in preparation for computing the A_i RHS. First, we aim to compute
         \partial_t A_z - [gauge terms] = \psi^{6} (v^x B^y - v^y B^x)
     where A_z is defined at (i+1/2, j+1/2, k). For this, we need vx, vy, Bx, and By. Since
     we have B_stagger, the main issue is the double reconstructed velocities. However, we
     saved the previous reconstructions for this. So, the following reconstructs
         Bx_stagger @ (i+1/2, j, k) -> (i+1/2, j-1/2, k)
         By_stagger @ (i, j+1/2, k) -> (i-1/2, j+1/2, k)
         v{x,y}_{r,l} @ (i-1/2, j, k) -> (i-1/2, j-1/2, k)

      We also perform the reconstruction
         Bz_stagger @ (i, j, k+1/2) -> (i, j-1/2, k+1/2)
      in preparation for computing the RHS of A_x.
  */
  {
    const int var_indices[1] = {BY_STAGGER};
    reconstruction_loop(flux_dir, eos, 1, var_indices, rho_b, pressure, vel[flux_dir], in_prims, out_prims_r, out_prims_l);
  }

  flux_dir=1;

  {
    const int var_indices[6] = {VXR, VYR, VXL, VYL, BX_STAGGER, BZ_STAGGER};
    reconstruction_loop(flux_dir, eos, 6, var_indices, rho_b, pressure, vel[flux_dir], in_prims, out_prims_r, out_prims_l);
  }

  /*
     We compute the y-direction RHS for the conservative hydro variables using PPM
     reconstruction from (i, j, k) to (i, j-1/2, k). This reconstructs rho_b, P, v^i, and
     B^i. B^x is not reconstructed, as this is given by the Bx_stagger grid function. This
     is all done to be able to compute
         \partial_x F = [ F(i,j+1/2,k) - F(i,j-1/2,k) ] / dx
     We also save the v^i reconstructions for the A_i RHS.
  */
  calculate_MHD_dirn_rhs(flux_dir, eos, dX, B_center, B_stagger[flux_dir],
                         vel_r, vel_l, cmin[flux_dir], cmax[flux_dir],
                         auxevol_gfs, in_gfs, rhs_gfs);

  /*****************************************
   * COMPUTING RHS OF A_z, BOOKKEEPING NOTE:
   * We want to compute
   * \partial_t A_z - [gauge terms] = \psi^{6} (v^x B^y - v^y B^x).
   * A_z is defined at (i+1/2,j+1/2,k).
   * ==========================
   * Where defined  | Variables
   * (i-1/2,j-1/2,k)| {vxrr,vxrl,vxlr,vxll,vyrr,vyrl,vylr,vyll}
   * (i+1/2,j-1/2,k)| {Bx_stagger_r,Bx_stagger_l} (see Table 1 in arXiv:1007.2848)
   * (i-1/2,j+1/2,k)| {By_stagger_r,By_stagger_l} (see Table 1 in arXiv:1007.2848)
   * (i,j,k)        | {phi}
   * ==========================
   ******************************************/
  flux_dir=2;
  A_flux_rhs(flux_dir, out_prims_r, out_prims_l, phi_bssn, cmin, cmax, rhs_gfs);

  /*
     Here we perform reconstructions in preparation for computing the A_i RHS. We aim to compute
         \partial_t A_x - [gauge terms] = \psi^{6} (v^y B^z - v^z B^y)
     where A_x is defined at (i, j+1/2, k+1/2). For this, we need vy, vz, By, and Bz. We previously reconstructed
         Bz_stagger @ (i, j, k+1/2) -> (i, j-1/2, k+1/2)

     The following reconstructs
         By_stagger @ (i, j+1/2, k) -> (i, j+1/2, k-1/2)
         v{y,z}_{r,l} @ (i-1/2, j, k) -> (i-1/2, j-1/2, k)

      We also perform the reconstruction
         Bx_stagger @ (i+1/2, j, k) -> (i+1/2, j, k-1/2)
      in preparation for computing the RHS of A_y.
  */
  {
    const int var_indices[6] = {VYR, VZR, VYL, VZL, BX_STAGGER, BY_STAGGER};
    reconstruction_loop(flux_dir, eos, 6, var_indices, rho_b, pressure, vel[flux_dir], in_prims, out_prims_r, out_prims_l);
  }

  /*
     We compute the z-direction RHS for the conservative hydro variables using PPM
     reconstruction from (i, j, k) to (i, j-1/2, k). This reconstructs rho_b, P, v^i, and
     B^i. B^x is not reconstructed, as this is given by the Bx_stagger grid function. This
     is all done to be able to compute
       \partial_z F = [ F(i,j,k+1/2) - F(i,j,k-1/2) ] / dx
     We also save the v^i reconstructions for the A_i RHS.
  */
  calculate_MHD_dirn_rhs(flux_dir, eos, dX, B_center, B_stagger[flux_dir],
                         vel_r, vel_l, cmin[flux_dir], cmax[flux_dir],
                         auxevol_gfs, in_gfs, rhs_gfs);

  /*****************************************
   * COMPUTING RHS OF A_x, BOOKKEEPING NOTE:
   * We want to compute
   * \partial_t A_x - [gauge terms] = \psi^{6} (v^y B^z - v^z B^y).
   * A_x is defined at (i,j+1/2,k+1/2).
   * ==========================
   * Where defined  | Variables
   * (i,j-1/2,k-1/2)| {vyrr,vyrl,vylr,vyll,vzrr,vzrl,vzlr,vzll}
   * (i,j+1/2,k-1/2)| {By_stagger_r,By_stagger_l} (see Table 1 in arXiv:1007.2848)
   * (i,j-1/2,k+1/2)| {Bz_stagger_r,Bz_stagger_l} (see Table 1 in arXiv:1007.2848)
   * (i,j,k)        | {phi}
   * ==========================
   ******************************************/
  flux_dir=0;
  // cmin/max could be done internally using the same indices as v and B if all c var pointers were collected into
  // single array
  A_flux_rhs(flux_dir, out_prims_r, out_prims_l, phi_bssn, cmin, cmax, rhs_gfs);

  /*
     Here we perform reconstructions in preparation for computing the A_i RHS. We aim to compute
         \partial_t A_y - [gauge terms] = \psi^{6} (v^z B^x - v^x B^z)
     where A_y is defined at (i+1/2, j, k+1/2). For this, we need vx, vz, Bx, and Bz. We previously reconstructed
         Bx_stagger @ (i+1/2, j, k) -> (i+1/2, j, k-1/2)
     The following reconstructs
         Bz_stagger @ (i, j, k+1/2) -> (i-1/2, j, k+1/2)
         v{x,z}_{r,l} @ (i, j, k-1/2) -> (i-1/2, j, k-1/2)
  */
  {
    const int var_indices[5] = {VXR, VZR, VXL, VZL, BZ_STAGGER};
    reconstruction_loop(flux_dir, eos, 5, var_indices, rho_b, pressure, vel[flux_dir], in_prims, out_prims_r, out_prims_l);
  }


  /*****************************************
   * COMPUTING RHS OF A_y, BOOKKEEPING NOTE:
   * We want to compute
   * \partial_t A_y - [gauge terms] = \psi^{6} (v^z B^x - v^x B^z).
   * A_y is defined at (i+1/2,j,k+1/2).
   * ==========================
   * Where defined  | Variables
   * (i-1/2,j,k-1/2)| {vxrr,vxrl,vxlr,vxll,vzrr,vzrl,vzlr,vzll}
   * (i+1/2,j,k-1/2)| {Bx_stagger_r,Bx_stagger_l} (see Table 1 in arXiv:1007.2848)
   * (i-1/2,j,k+1/2)| {Bz_stagger_r,Bz_stagger_l} (see Table 1 in arXiv:1007.2848)
   * (i,j,k)        | {phi}
   * ==========================
   ******************************************/
  flux_dir=1;
  A_flux_rhs(flux_dir, out_prims_r, out_prims_l, phi_bssn, cmin, cmax, rhs_gfs);

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [19]:
def add_to_Cfunction_dict__reconstruction_loop():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "reconstruction_loop"
    params = r"""
const int flux_dir,
const eos_parameters *restrict ghl_eos,
const int num_vars,
const int *restrict var_indices,
const double *rho_b,
const double *pressure,
const double *v_flux,
const double **in_prims,
double **out_prims_r,
double **out_prims_l"""

    prefunc = r"""
REAL eos_Gamma_eff(const eos_parameters *restrict ghl_eos, const REAL rho_in, const REAL press_in) {
  REAL K, Gamma;
  ghl_hybrid_get_K_and_Gamma(ghl_eos, rho_in, &K, &Gamma);
  const REAL P_cold = K*pow(rho_in, Gamma);
  return eos->Gamma_th + (Gamma - eos->Gamma_th)*P_cold/press_in;
}
"""

    body = r"""

  const int xdir = (flux_dir == 0);
  const int ydir = (flux_dir == 1);
  const int zdir = (flux_dir == 2);

  const int imin = NGHOSTS;
  const int jmin = NGHOSTS;
  const int kmin = NGHOSTS;
  const int imax = Nxx_plus_2NGHOSTS0 - NGHOSTS + 1;
  const int jmax = Nxx_plus_2NGHOSTS1 - NGHOSTS + 1;
  const int kmax = Nxx_plus_2NGHOSTS2 - NGHOSTS + 1;

#pragma omp parallel for
  for(int k=kmin; k<kmax; k++) {
    for(int j=jmin; j<jmax; j++) {
      for(int i=imin; i<imax; i++) {
        const int index = IDX3S(i, j, k);
        double press_stencil[6], v_flux_stencil[6];
        double var_data[num_vars][6], vars_r[num_vars], vars_l[num_vars];

        for(int ind=0; ind<6; ind++) {
          // Stencil from -3 to +2 reconstructs to e.g. i-1/2
          const int stencil = IDX3S(i+xdir*(ind-3), j+ydir*(ind-3), k+zdir*(ind-3));
          v_flux_stencil[ind] = v_flux[stencil]; // Could be smaller; doesn't use full stencil
          press_stencil[ind] = pressure[stencil];
          for(int var=0; var<num_vars; var++) {
            var_data[var][ind] = in_prims[var_indices[var]][stencil];
          }
        }

        // Compute Gamma
        const double Gamma = eos_Gamma_eff(ghl_eos, rho_b[index], pressure[index]);

        ghl_ppm_no_rho_P(
              press_stencil, var_data,
              num_vars, v_flux_stencil, Gamma,
              vars_r, vars_l);

        for(int var=0; var<num_vars; var++) {
          out_prims_r[var_indices[var]][index] = vars_r[var];
          out_prims_l[var_indices[var]][index] = vars_l[var];
        }
      }
    }
  }

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        prefunc=prefunc,
        loopopts = "")

In [20]:
def add_to_Cfunction_dict__prims_to_conservs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "prims_to_conservs"
    params = r"""
const paramstruct *restrict params,
const eos_parameters *restrict eos,
REAL *restrict in_gfs,
REAL *restrict auxevol_gfs"""

    desc = r"""
/********************************
 *
 * Written in 2014 by Zachariah B. Etienne
 *
 * Sets metric & MHD variables needed
 * by IllinoisGRMHD, converting from
 * HydroBase and ADMBase.
 ********************************/
"""

    body = r"""

  const double poison = 0.0/0.0;
  double dummy0, dummy1, dummy2, dummy3;

  const int imax = Nxx_plus_2NGHOSTS0 - NGHOSTS;
  const int jmax = Nxx_plus_2NGHOSTS1 - NGHOSTS;
  const int kmax = Nxx_plus_2NGHOSTS2 - NGHOSTS;

#pragma omp parallel for
  for(int k=0; k<kmax; k++) {
    for(int j=0; j<jmax; j++) {
      for(int i=0; i<imax; i++) {
        const int index = IDX3S(i,j,k);

        ghl_metric_quantities ADM_metric;
        ghl_initialize_metric(auxevol_gfs[IDX4ptS(ALPHAGF, index)],
                              auxevol_gfs[IDX4ptS(BETAU0GF, index)],
                              auxevol_gfs[IDX4ptS(BETAU1GF, index)],
                              auxevol_gfs[IDX4ptS(BETAU2GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD00GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD01GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD02GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD11GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD12GF, index)],
                              auxevol_gfs[IDX4ptS(GAMMADD22GF, index)],
                              &ADM_metric);

        ghl_ADM_aux_quantities metric_aux;
        ghl_compute_ADM_auxiliaries(&ADM_metric, &metric_aux);

        ghl_primitive_quantities prims;
        ghl_initialize_primitives(
              auxevol_gfs[IDX4ptS(RHOBGF, index)],
              auxevol_gfs[IDX4ptS(PGF, index)],
              poison,
              auxevol_gfs[IDX4ptS(VU0GF, index)],
              auxevol_gfs[IDX4ptS(VU1GF, index)],
              auxevol_gfs[IDX4ptS(VU2GF, index)],
              auxevol_gfs[IDX4ptS(B_CENTERU0GF, index)],
              auxevol_gfs[IDX4ptS(B_CENTERU1GF, index)],
              auxevol_gfs[IDX4ptS(B_CENTERU2GF, index)],
              poison, poison, poison, // entropy, Y_e, temp
              &prims);

        ghl_conservative_quantities cons;
        int speed_limited = 0;
        //This applies inequality fixes on the primitives
        ghl_enforce_primitive_limits_and_compute_u0(
              ghl_params, ghl_eos, &ADM_metric,
              &prims, &speed_limited);
        //This computes the conservatives and stress-energy tensor from the new primitives
        ghl_compute_conservs(
              &ADM_metric, &metric_aux, &prims, &cons);

        ghl_return_primitives(
              &prims,
              &auxevol_gfs[IDX4ptS(RHOBGF, index)],
              &auxevol_gfs[IDX4ptS(PGF, index)],
              &dummy0,
              &auxevol_gfs[IDX4ptS(VU0GF, index)],
              &auxevol_gfs[IDX4ptS(VU1GF, index)],
              &auxevol_gfs[IDX4ptS(VU2GF, index)],
              &auxevol_gfs[IDX4ptS(B_CENTERU0GF, index)],
              &auxevol_gfs[IDX4ptS(B_CENTERU1GF, index)],
              &auxevol_gfs[IDX4ptS(B_CENTERU2GF, index)],
              &dummy1, &dummy2, &dummy3);

        ghl_return_conservatives(&cons,
          &in_gfs[IDX4ptS(RHO_STARGF, index)],
          &in_gfs[IDX4ptS(TAU_TILDEGF, index)],
          &in_gfs[IDX4ptS(STILDED0GF, index)],
          &in_gfs[IDX4ptS(STILDED1GF, index)],
          &in_gfs[IDX4ptS(STILDED2GF, index)],
          &dummy1, &dummy2);

      }
    }
  }

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [21]:
def add_to_Cfunction_dict__evaluate_tau_curvature_rhs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "evaluate_tau_curvature_rhs"
    params = r"""
const paramstruct *restrict params,
const eos_parameters *restrict ghl_eos,
const REAL *restrict in_gfs,
const REAL *restrict auxevol_gfs,
REAL *restrict rhs_gfs"""

    body = r"""

  const double poison = 0.0/0.0;

  const int imin = NGHOSTS;
  const int jmin = NGHOSTS;
  const int kmin = NGHOSTS;
  const int imax = Nxx_plus_2NGHOSTS0 - NGHOSTS;
  const int jmax = Nxx_plus_2NGHOSTS1 - NGHOSTS;
  const int kmax = Nxx_plus_2NGHOSTS2 - NGHOSTS;

#pragma omp parallel for
  for(int k=kmin; k<kmax; k++) {
    for(int j=jmin; j<jmax; j++) {
      for(int i=imin; i<imax; i++) {
        const int index = CCTK_GFINDEX3D(cctkGH, i, j ,k);

        // Initialize RHS variables to zero
        rhs_gfs[IDX4ptS(RHO_STARGF, index)]  = 0.0;
        rhs_gfs[IDX4ptS(STILDED0GF, index)]  = 0.0;
        rhs_gfs[IDX4ptS(STILDED1GF, index)]  = 0.0;
        rhs_gfs[IDX4ptS(STILDED2GF, index)]  = 0.0;
        rhs_gfs[IDX4ptS(PHITILDE, index)]    = 0.0;
        rhs_gfs[IDX4ptS(AD0GF, index)]       = 0.0;
        rhs_gfs[IDX4ptS(AD1GF, index)]       = 0.0;
        rhs_gfs[IDX4ptS(AD2GF, index)]       = 0.0;

        ghl_metric_quantities ADM_metric;
        ghl_initialize_metric(
              ghl_initialize_metric(auxevol_gfs[IDX4ptS(ALPHAGF, index)],
              auxevol_gfs[IDX4ptS(BETAU0GF, index)],
              auxevol_gfs[IDX4ptS(BETAU1GF, index)],
              auxevol_gfs[IDX4ptS(BETAU2GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD00GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD01GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD02GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD11GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD12GF, index)],
              auxevol_gfs[IDX4ptS(GAMMADD22GF, index)],
              &ADM_metric);

        ghl_extrinsic_curvature curv;
        ghl_initialize_extrinsic_curvature(
              auxevol_gfs[IDX4ptS(KDD00GF, index)],
              auxevol_gfs[IDX4ptS(KDD01GF, index)],
              auxevol_gfs[IDX4ptS(KDD02GF, index)],
              auxevol_gfs[IDX4ptS(KDD11GF, index)],
              auxevol_gfs[IDX4ptS(KDD12GF, index)],
              auxevol_gfs[IDX4ptS(KDD22GF, index)],
              &curv);

        ghl_primitive_quantities prims;
        ghl_initialize_primitives(
              auxevol_gfs[IDX4ptS(RHOBGF, index)],
              auxevol_gfs[IDX4ptS(PGF, index)],
              poison,
              auxevol_gfs[IDX4ptS(VU0GF, index)],
              auxevol_gfs[IDX4ptS(VU1GF, index)],
              auxevol_gfs[IDX4ptS(VU2GF, index)],
              auxevol_gfs[IDX4ptS(B_CENTERU0GF, index)],
              auxevol_gfs[IDX4ptS(B_CENTERU1GF, index)],
              auxevol_gfs[IDX4ptS(B_CENTERU2GF, index)],
              poison, poison, poison, // entropy, Y_e, temp
              &prims);

        int speed_limited = 0;
        ghl_limit_v_and_compute_u0(
              ghl_eos, &ADM_metric, &prims, &speed_limited);

        ghl_conservative_quantities cons_source;
        cons_source.tau = 0;
        ghl_calculate_tau_tilde_source_term_extrinsic_curv(
                 &prims, ghl_eos, &ADM_metric, &curv, &cons_source);
        rhs_gfs[IDX4ptS(TAU_TILDEGF, index)] = cons_source.tau;
    }
  }
}

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [22]:
def add_to_Cfunction_dict__evaluate_phitilde_and_A_gauge_rhs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "evaluate_phitilde_and_A_gauge_rhs"
    params = r"""
const paramstruct *restrict params,
const eos_parameters *restrict ghl_eos,
const REAL *restrict evol_gfs,
const REAL *restrict auxevol_gfs,
REAL *restrict rhs_gfs"""

    body = r"""

  // Note that in this function, we don't bother with reconstruction, instead interpolating.
  // We need A^i, but only have A_i. So we use the BSSN metric gtupij.
  // The reconstruction variables are temporary variables and the data in them can be safely overwritten,
  // saving some memory.
  double *betax_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU0GF;
  double *betay_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU1GF;
  double *betaz_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRU2GF;
  double *alpha_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VRRU0GF;
  double *alpha_Phi_minus_betaj_A_j_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLLU0GF;
  double *sqrtg_Ax_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU0GF;
  double *sqrtg_Ay_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU1GF;
  double *sqrtg_Az_interp = auxevol_gfs + Nxx_plus_2NGHOSTS_TOT*VLU2GF;

  /* Compute \partial_t psi6phi = -\partial_i (  \alpha psi^6 A^i - psi6phi \beta^i)
   *    (Eq 13 of http://arxiv.org/pdf/1110.4633.pdf), using Lorenz gauge.
   * Note that the RHS consists of a shift advection term on psi6phi and
   *    a term depending on the vector potential.
   * psi6phi is defined at (i+1/2,j+1/2,k+1/2), but instead of reconstructing
   *    to compute the RHS of \partial_t psi6phi, we instead use standard
   *    interpolations.
   */

  // We declare these values to be over the interior so the setting of ghostzone points is
  // more transparent.
  const int imin = NGHOSTS;
  const int jmin = NGHOSTS;
  const int kmin = NGHOSTS;
  const int imax = Nxx_plus_2NGHOSTS0 - NGHOSTS;
  const int jmax = Nxx_plus_2NGHOSTS1 - NGHOSTS;
  const int kmax = Nxx_plus_2NGHOSTS2 - NGHOSTS;

  // The RHS loop requires 2 ghostzones, so this loop must set those values, hence this loop
  // goes into the ghostzones. This loop requires a stencil of {-1,1},{-1,1},{-1,1},
  // which uses the last of the 3 ghostzones required by the simulation.
  // Note that ALL input variables are defined at ALL gridpoints, so no
  // worries about ghostzones.
#pragma omp parallel for
  for(int k=kmin-2; k<kmax+2; k++) {
    for(int j=jmin-2; j<jmax+2; j++) {
      for(int i=imin-2; i<imax+2; i++) {
        const int index=IDX3S(i,j,k);

        // First compute \partial_j \alpha \sqrt{\gamma} A^j (RHS of \partial_i psi6phi)
        // FIXME: Would be much cheaper & easier to unstagger A_i, raise, then interpolate A^i.
        //        However, we keep it this way to be completely compatible with the original
        //        Illinois GRMHD thorn, called mhd_evolve.
        //
        //Step 1) j=x: Need to raise A_i, but to do that, we must have all variables at the same gridpoints:
        // The goal is to compute \partial_j (\alpha \sqrt{\gamma} A^j) at (i+1/2,j+1/2,k+1/2)
        //    We do this by first interpolating (RHS1x) = (\alpha \sqrt{\gamma} A^x) at
        //    (i,j+1/2,k+1/2)and (i+1,j+1/2,k+1/2), then taking \partial_x (RHS1x) =
        //    [ RHS1x(i+1,j+1/2,k+1/2) - RHS1x(i,j+1/2,k+1/2) ]/dX.
        // First bring gtup's, psi, and alpha to (i,j+1/2,k+1/2):
        ghl_metric_quantities metric_stencil[2][2][2];
        double Ax_stencil[3][3][3];
        double Ay_stencil[3][3][3];
        double Az_stencil[3][3][3];
        induction_interp_vars interp_vars;

        // Read in variable at interpolation stencil points from main memory.
        for(int iterz=0; iterz<2; iterz++)
          for(int itery=0; itery<2; itery++)
            for(int iterx=0; iterx<2; iterx++) {
              const int ind = IDX3S(i+iterx,j+itery,k+iterz);

              ghl_initialize_metric(auxevol_gfs[IDX4ptS(ALPHAGF, ind)],
                              auxevol_gfs[IDX4ptS(BETAU0GF, ind)],
                              auxevol_gfs[IDX4ptS(BETAU1GF, ind)],
                              auxevol_gfs[IDX4ptS(BETAU2GF, ind)],
                              auxevol_gfs[IDX4ptS(GAMMADD00GF, ind)],
                              auxevol_gfs[IDX4ptS(GAMMADD01GF, ind)],
                              auxevol_gfs[IDX4ptS(GAMMADD02GF, ind)],
                              auxevol_gfs[IDX4ptS(GAMMADD11GF, ind)],
                              auxevol_gfs[IDX4ptS(GAMMADD12GF, ind)],
                              auxevol_gfs[IDX4ptS(GAMMADD22GF, ind)],
                    &metric_stencil[iterz][itery][iterx]);
        }
        // A_x needs a stencil s.t. interp_limits={ 0,1,-1,1,-1,1}.
        // A_y needs a stencil s.t. interp_limits={-1,1, 0,1,-1,1}.
        // A_z needs a stencil s.t. interp_limits={-1,1,-1,1, 0,1}.
        // We could fill only the needed elements, but it is cleaner
        // to fill in the whole 3x3x3 array.
        // TODO: the old code explicitly only filled in the necessary
        // elements. If we want to remove ~15 memcopies, do that here.
        for(int iterz=-1; iterz<2; iterz++)
          for(int itery=-1; itery<2; itery++)
            for(int iterx=-1; iterx<2; iterx++) {
              const int ind = IDX3S(i+iterx,j+itery,k+iterz);
              Ax_stencil[iterz+1][itery+1][iterx+1] = evol_gfs[IDX4ptS(AD0GF, ind)];
              Ay_stencil[iterz+1][itery+1][iterx+1] = evol_gfs[IDX4ptS(AD1GF, ind)];
              Az_stencil[iterz+1][itery+1][iterx+1] = evol_gfs[IDX4ptS(AD2GF, ind)];
        }
// This code should only copy the needed data that isn't copied in the loop for other variables, but it is untested.
//        for(int iter2=0; iter2<2; iter2++)
//        for(int iter1=0; iter1<2; iter1++) {
//          gauge_vars.A_x[iter2+1][0][iter1+1] = in_vars[A_XI][IDX3S( i+iter1,     j-1, k+iter2)]; // { (0,1),    -1, (0,1)}
//          gauge_vars.A_x[0][iter2+1][iter1+1] = in_vars[A_XI][IDX3S( i+iter1, j+iter2, k-1    )]; // { (0,1), (0,1), -1}
//          gauge_vars.A_y[iter2+1][iter1+1][0] = in_vars[A_YI][IDX3S(     i-1, j+iter1, k+iter2)]; // { -1,    (0,1), (0,1)}
//          gauge_vars.A_y[0][iter1+1][iter2+1] = in_vars[A_YI][IDX3S( i+iter2, j+iter1, k-1    )]; // { (0,1), (0,1), -1}
//          gauge_vars.A_z[iter1+1][iter2+1][0] = in_vars[A_ZI][IDX3S(     i-1, j+iter2, k+iter1)]; // { -1,    (0,1), (0,1)}
//          gauge_vars.A_z[iter1+1][0][iter2+1] = in_vars[A_ZI][IDX3S( i+iter2,     j-1, k+iter1)]; // { (0,1),    -1, (0,1)}
//        }

        ghl_interpolate_with_cell_centered_ADM(metric_stencil, Ax_stencil, Ay_stencil, Az_stencil, evol_gfs[IDX4ptS(PHITILDEGF, index)], &interp_vars);

        alpha_interp[index] = interp_vars.alpha;
        sqrtg_Ax_interp[index] = interp_vars.sqrtg_Ai[0];
        sqrtg_Ay_interp[index] = interp_vars.sqrtg_Ai[1];
        sqrtg_Az_interp[index] = interp_vars.sqrtg_Ai[2];
        alpha_Phi_minus_betaj_A_j_interp[index] = interp_vars.alpha_Phi_minus_betaj_A_j;
        betax_interp[index] = interp_vars.betai[0];
        betay_interp[index] = interp_vars.betai[1];
        betaz_interp[index] = interp_vars.betai[2];
      }
    }
  }

  const REAL dxi[3] = { 1.0/dxx0, 1.0/dxx1, 1.0/dxx2 };

#pragma omp parallel for
  for(int k=kmin; k<kmax; k++) {
    for(int j=jmin; j<jmax; j++) {
      for(int i=imin; i<imax; i++) {
        const int index = IDX3S(i,j,k);

        // \partial_t A_i = [reconstructed stuff] + [gauge stuff],
        //    where [gauge stuff] = -\partial_i (\alpha \Phi - \beta^j A_j)
        rhs_gfs[IDX4ptS(AD0GF, index)] += dxi[0]*(alpha_Phi_minus_betaj_A_j_interp[IDX3S(i-1,j,k)] - alpha_Phi_minus_betaj_A_j_interp[index]);
        rhs_gfs[IDX4ptS(AD1GF, index)] += dxi[1]*(alpha_Phi_minus_betaj_A_j_interp[IDX3S(i,j-1,k)] - alpha_Phi_minus_betaj_A_j_interp[index]);
        rhs_gfs[IDX4ptS(AD2GF, index)] += dxi[2]*(alpha_Phi_minus_betaj_A_j_interp[IDX3S(i,j,k-1)] - alpha_Phi_minus_betaj_A_j_interp[index]);

        double betax_stencil[5], betay_stencil[5], betaz_stencil[5];
        double phitilde_stencil[3][5], sqrtg_Ai_stencil[3][2];

        sqrtg_Ai_stencil[0][0] = sqrtg_Ax_interp[index];
        sqrtg_Ai_stencil[1][0] = sqrtg_Ay_interp[index];
        sqrtg_Ai_stencil[2][0] = sqrtg_Az_interp[index];

        sqrtg_Ai_stencil[0][1] = sqrtg_Ax_interp[IDX3S(i+1,j,k)];
        sqrtg_Ai_stencil[1][1] = sqrtg_Ay_interp[IDX3S(i,j+1,k)];
        sqrtg_Ai_stencil[2][1] = sqrtg_Az_interp[IDX3S(i,j,k+1)];

        for(int iter=-2; iter<3; iter++) {
          const int indexx = IDX3S(i+iter,j,     k     );
          const int indexy = IDX3S(i,     j+iter,k     );
          const int indexz = IDX3S(i,     j,     k+iter);
          betax_stencil[iter+2] = betax_interp[indexx];
          betay_stencil[iter+2] = betay_interp[indexy];
          betaz_stencil[iter+2] = betaz_interp[indexz];
          phitilde_stencil[0][iter+2] = evol_gfs[IDX4ptS(PHITILDEGF, indexx)];
          phitilde_stencil[1][iter+2] = evol_gfs[IDX4ptS(PHITILDEGF, indexy)];
          phitilde_stencil[2][iter+2] = evol_gfs[IDX4ptS(PHITILDEGF, indexz)];
        }
        rhs_gfs[IDX4ptS(PHITILDEGF, index)] += ghl_calculate_phitilde_rhs(dxi, ghl_params->Lorenz_damping_factor, alpha_interp[index], betax_stencil, betay_stencil, betaz_stencil, sqrtg_Ai_stencil, phitilde_stencil);
      }
    }
  }

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [23]:
def add_to_Cfunction_dict__compute_B_and_Bstagger_from_A():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    c_type = "void"
    name = "compute_B_and_Bstagger_from_A"
    params = r"""
const paramstruct *restrict params,
const REAL *restrict Ax,
const REAL *restrict Ay,
const REAL *restrict Az,
const REAL *restrict gxx,
const REAL *restrict gxy,
const REAL *restrict gxz,
const REAL *restrict gyy,
const REAL *restrict gyz,
const REAL *restrict gzz,
REAL *restrict Bx_center,
REAL *restrict By_center,
REAL *restrict Bz_center,
REAL *restrict Bx_stagger,
REAL *restrict By_stagger,
REAL *restrict Bz_stagger"""

    body = r"""
  REAL dxi = 1.0/dxx0;
  REAL dyi = 1.0/dxx1;
  REAL dzi = 1.0/dxx2;

#pragma omp parallel for
 for(int k=0; k<Nxx_plus_2NGHOSTS2; k++) {
    for(int j=0; j<Nxx_plus_2NGHOSTS1; j++) {
      for(int i=0; i<Nxx_plus_2NGHOSTS0; i++) {
        // Look Mom, no if() statements!
        const int shiftedim1 = (i-1)*(i!=0); // This way, i=0 yields shiftedim1=0 and shiftedi=1, used below for our COPY boundary condition.
        const int shiftedi   = shiftedim1+1;

        const int shiftedjm1 = (j-1)*(j!=0);
        const int shiftedj   = shiftedjm1+1;

        const int shiftedkm1 = (k-1)*(k!=0);
        const int shiftedk   = shiftedkm1+1;

        int index,indexim1,indexjm1,indexkm1;

        const int actual_index = IDX3S(i, j, k);

        // For the lower boundaries, the following applies a "copy"
        //    boundary condition on Bi_stagger where needed.
        //    E.g., Bx_stagger(i,jmin,k) = Bx_stagger(i,jmin+1,k)
        //    We find the copy BC works better than extrapolation.
        // For the upper boundaries, we do the following copy:
        //    E.g., Bx_stagger(imax+1,j,k) = Bx_stagger(imax,j,k)
        // Note that B_stagger is densitized, so we do not divide by psi^6.
        /**************/
        /* Bx_stagger */
        /**************/

        index    = IDX3S(i, shiftedj,   shiftedk);
        indexjm1 = IDX3S(i, shiftedjm1, shiftedk);
        indexkm1 = IDX3S(i, shiftedj,   shiftedkm1);
        // Set Bx_stagger = \partial_y A_z - partial_z A_y
        // "Grid" Ax(i,j,k) is actually Ax(i,j+1/2,k+1/2)
        // "Grid" Ay(i,j,k) is actually Ay(i+1/2,j,k+1/2)
        // "Grid" Az(i,j,k) is actually Az(i+1/2,j+1/2,k)
        // Therefore, the 2nd order derivative \partial_z A_y at (i+1/2,j,k) is:
        //          ["Grid" Ay(i,j,k) - "Grid" Ay(i,j,k-1)]/dZ
        Bx_stagger[actual_index] = (Az[index]-Az[indexjm1])*dyi - (Ay[index]-Ay[indexkm1])*dzi;

        /**************/
        /* By_stagger */
        /**************/

        index    = IDX3S(shiftedi,   j, shiftedk);
        indexim1 = IDX3S(shiftedim1, j, shiftedk);
        indexkm1 = IDX3S(shiftedi,   j, shiftedkm1);
        // Set By_stagger = \partial_z A_x - \partial_x A_z
        By_stagger[actual_index] = (Ax[index]-Ax[indexkm1])*dzi - (Az[index]-Az[indexim1])*dxi;

        /**************/
        /* Bz_stagger */
        /**************/

        index    = IDX3S(hiftedi,   shiftedj,   k);
        indexim1 = IDX3S(hiftedim1, shiftedj,   k);
        indexjm1 = IDX3S(hiftedi,   shiftedjm1, k);
        // Set Bz_stagger = \partial_x A_y - \partial_y A_x
        Bz_stagger[actual_index] = (Ay[index]-Ay[indexim1])*dxi - (Ax[index]-Ax[indexjm1])*dyi;
      }
    }
  }

#pragma omp parallel for
  for(int k=0; k<Nxx_plus_2NGHOSTS2; k++) {
    for(int j=0; j<Nxx_plus_2NGHOSTS1; j++) {
      for(int i=0; i<Nxx_plus_2NGHOSTS0; i++) {
        // Look Mom, no if() statements!
        const int shiftedim1 = (i-1)*(i!=0); // This way, i=0 yields shiftedim1=0 and shiftedi=1, used below for our COPY boundary condition.
        const int shiftedi   = shiftedim1+1;

        const int shiftedjm1 = (j-1)*(j!=0);
        const int shiftedj   = shiftedjm1+1;

        const int shiftedkm1 = (k-1)*(k!=0);
        const int shiftedk   = shiftedkm1+1;

        int index,indexim1,indexjm1,indexkm1;

        const int actual_index = IDX3S(i, j, k);

        // Unlike B_stagger, B_center is undensitized. Therefore, we
        // need to divide by psi^6 = \sqrt{gamma}
        const double sqrt_detgamma = sqrt(fabs(
                gxx[actual_index] * (gyy[actual_index]*gzz[actual_index] - gyz[actual_index]*gyz[actual_index])
              + gxy[actual_index] * (gyz[actual_index]*gxz[actual_index] - gxy[actual_index]*gzz[actual_index])
              + gxz[actual_index] * (gxy[actual_index]*gyz[actual_index] - gyy[actual_index]*gxz[actual_index])));

        // For the lower boundaries, the following applies a "copy"
        //    boundary condition on Bi and Bi_stagger where needed.
        //    E.g., Bx_center(imin,j,k) = Bx_center(imin+1,j,k)
        //    We find the copy BC works better than extrapolation.
        /*************/
        /* Bx_center */
        /*************/
        index = IDX3S(shiftedi, j, k);
        indexim1 = IDX3S(shiftedim1, j, k);
        // Set Bx_center = 0.5 ( Bx_stagger + Bx_stagger_im1 )
        // "Grid" Bx_stagger(i,j,k) is actually Bx_stagger(i+1/2,j,k)
        Bx_center[actual_index] = 0.5 * ( Bx_stagger[index] + Bx_stagger[indexim1] )/sqrt_detgamma;

        /*************/
        /* By_center */
        /*************/
        index = IDX3S(i, shiftedj, k);
        indexjm1 = IDX3S(i, shiftedjm1, k);
        // Set By_center = 0.5 ( By_stagger + By_stagger_im1 )
        // "Grid" By_stagger(i,j,k) is actually By_stagger(i,j+1/2,k)
        By_center[actual_index] = 0.5 * ( By_stagger[index] + By_stagger[indexjm1] )/sqrt_detgamma;

        /*************/
        /* Bz_center */
        /*************/
        index = IDX3S(i, j, shiftedk);
        indexkm1 = IDX3S(i, j, shiftedkm1);
        // Set Bz_center = 0.5 ( Bz_stagger + Bz_stagger_im1 )
        // "Grid" Bz_stagger(i,j,k) is actually Bz_stagger(i,j+1/2,k)
        Bz_center[actual_index] = 0.5 * ( Bz_stagger[index] + Bz_stagger[indexkm1] )/sqrt_detgamma;
      }
    }
  }

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [ ]:
def add_to_Cfunction_dict__A_flux_rhs():

    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]

    c_type = "void"
    name = "A_flux_rhs"
    params = r"""
const int flux_dir,
const double **in_prims_r,
const double **in_prims_l,
const double **cmin,
const double **cmax,
double *restrict A_rhs"""

    body = r"""

  const int xdir = (flux_dir==0);
  const int ydir = (flux_dir==1);
  const int zdir = (flux_dir==2);

  // These are used to determine which components of v,
  // B, cmin, and cmax are used in the computation.
  const int dir1_offset = (flux_dir+1)%3, dir2_offset = (flux_dir+2)%3;

  // This offsets the index by +1 in the perpendicular directions
  const int v_offset[3] = { !xdir, !ydir, !zdir };

  // This offsets the index by +1 in the permuted direction (x<-y<-z)
  const int B1_offset[3] = { ydir, zdir, xdir };

  // This offsets the index by +1 in the permuted direction (x->y->z)
  const int B2_offset[3] = { zdir, xdir, ydir };

  const int imin = NGHOSTS;
  const int jmin = NGHOSTS;
  const int kmin = NGHOSTS;
  const int imax = Nxx_plus_2NGHOSTS0 - NGHOSTS;
  const int jmax = Nxx_plus_2NGHOSTS1 - NGHOSTS;
  const int kmax = Nxx_plus_2NGHOSTS2 - NGHOSTS;

#pragma omp parallel for
  for(int k=kmin; k<kmax; k++) {
    for(int j=jmin; j<jmax; j++) {
      for(int i=imin; i<imax; i++) {
        const int index    = IDX3S(i,j,k);
        const int index_v  = IDX3S(i+v_offset[0], j+v_offset[1], k+v_offset[2]);
        const int index_B1 = IDX3S(i+B1_offset[0],j+B1_offset[1],k+B1_offset[2]);
        const int index_B2 = IDX3S(i+B2_offset[0],j+B2_offset[1],k+B2_offset[2]);

        HLL_2D_vars vars;

        vars.v1rr = in_prims_r[VXR+dir1_offset][index_v];
        vars.v1rl = in_prims_l[VXR+dir1_offset][index_v];
        vars.v1lr = in_prims_r[VXL+dir1_offset][index_v];
        vars.v1ll = in_prims_l[VXL+dir1_offset][index_v];

        vars.v2rr = in_prims_r[VXR+dir2_offset][index_v];
        vars.v2rl = in_prims_l[VXR+dir2_offset][index_v];
        vars.v2lr = in_prims_r[VXL+dir2_offset][index_v];
        vars.v2ll = in_prims_l[VXL+dir2_offset][index_v];

        vars.B1r = in_prims_r[BX_STAGGER+dir1_offset][index_B1];
        vars.B1l = in_prims_l[BX_STAGGER+dir1_offset][index_B1];

        vars.B2r = in_prims_r[BX_STAGGER+dir2_offset][index_B2];
        vars.B2l = in_prims_l[BX_STAGGER+dir2_offset][index_B2];

        /*
          Note that cmax/cmin (\alpha^{\pm}  as defined in Del Zanna et al) is at a slightly DIFFERENT
          point (e.g., (i+1/2,j,k) instead of (i+1/2,j+1/2,k) for F3). Yuk Tung Liu discussed this point
          with M. Shibata, who found that the effect is negligible.
        */
        vars.c1_min = cmin[dir1_offset][index_B2];
        vars.c1_max = cmax[dir1_offset][index_B2];
        vars.c2_min = cmin[dir2_offset][index_B1];
        vars.c2_max = cmax[dir2_offset][index_B1];

        A_rhs[index] = ghl_HLL_2D_flux_with_Btilde(&vars);
      }
    }
  }

"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        name=name, params=params,
        body=body,
        loopopts = "")

In [10]:
def add_to_Cfunction_dict__conservs_to_prims():

    desc = r"""
We evolve forward in time a set of functions called the
"conservative variables", and any time the conserv's
are updated, we must solve for the primitive variables
(rho, pressure, velocities) using a Newton-Raphson
technique, before reconstructing & evaluating the RHSs
of the MHD equations again.

This file contains the driver routine for this Newton-
Raphson solver. Truncation errors in conservative
variables can lead to no physical solutions in
primitive variables. We correct for these errors here
through a number of tricks described in the appendices
of http://arxiv.org/pdf/1112.0568.pdf.

This is a wrapper for the 2d solver of Noble et al. See
harm_utoprim_2d.c for references and copyright notice
for that solver. This wrapper was primarily written by
Zachariah Etienne & Yuk Tung Liu, in 2011-2013.

For optimal compatibility, this wrapper is licensed under
the GPL v2 or any later version.

Note that this code assumes a simple gamma law for the
moment, though it would be easy to extend to a piecewise
polytrope.
"""

    name = "cons_to_prims"
    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h"]
    params = r"""
const paramstruct *restrict params,
const ghl_parameters *restrict ghl_params,
const eos_parameters *restrict eos,
REAL * evol_gfs,
REAL *auxevol_gfs
"""

    body = r"""
  // These BSSN-based variables are not evolved, and so are not defined anywhere that the grid has moved.
  // Here we convert ADM variables (from ADMBase) to the BSSN-based variables expected by this routine.
  //IllinoisGRMHD_convert_ADM_to_BSSN__enforce_detgtij_eq_1__and_compute_gtupij(cctkGH,cctk_lsh,  gxx,gxy,gxz,gyy,gyz,gzz,alp,
  //                                                              gtxx,gtxy,gtxz,gtyy,gtyz,gtzz,
  //                                                              gtupxx,gtupxy,gtupxz,gtupyy,gtupyz,gtupzz,
  //                                                              phi_bssn,psi_bssn,lapm1);

  double poison = 1e200;

  // Diagnostic variables.
  int failures=0;
  int vel_limited_ptcount=0;
  int atm_resets=0;
  int rho_star_fix_applied=0;
  int pointcount=0;
  int failures_inhoriz=0;
  int pointcount_inhoriz=0;
  int backup0=0;
  int backup1=0;
  int backup2=0;
  int nan_found=0;
  double error_int_numer=0;
  double error_int_denom=0;
  int n_iter=0;
  double dummy0, dummy1, dummy2, dummy3, dummy4, dummy5, dummy6;

#pragma omp parallel for reduction(+:failures,vel_limited_ptcount,atm_resets,rho_star_fix_applied,pointcount,failures_inhoriz,pointcount_inhoriz,backup0,backup1,backup2,nan_found,error_int_numer,error_int_denom,n_iter) schedule(static)
  for(int k=NGHOSTS;k<Nxx_plus_2NGHOSTS2-NGHOSTS;k++)
    for(int j=NGHOSTS;j<Nxx_plus_2NGHOSTS1-NGHOSTS;j++)
      for(int i=NGHOSTS;i<Nxx_plus_2NGHOSTS0-NGHOSTS;i++) {
        const int index = IDX3S(i,j,k);

     con2prim_diagnostics diagnostics;
     ghl_initialize_diagnostics(&diagnostics);

    // Read in ADM metric quantities from gridfunctions and
    // set auxiliary and ADM metric quantities
    metric_quantities ADM_metric;
    ghl_initialize_metric(
        auxevol_gfs[IDX4ptS(ALPHAGF, index)],
        auxevol_gfs[IDX4ptS(BETAU0GF, index)],
        auxevol_gfs[IDX4ptS(BETAU1GF, index)],
        auxevol_gfs[IDX4ptS(BETAU2GF, index)],
        auxevol_gfs[IDX4ptS(GAMMADD00GF, index)],
        auxevol_gfs[IDX4ptS(GAMMADD01GF, index)],
        auxevol_gfs[IDX4ptS(GAMMADD02GF, index)],
        auxevol_gfs[IDX4ptS(GAMMADD11GF, index)],
        auxevol_gfs[IDX4ptS(GAMMADD12GF, index)],
        auxevol_gfs[IDX4ptS(GAMMADD22GF, index)],
          &ADM_metric);

    ADM_aux_quantities metric_aux;
    ghl_compute_ADM_auxiliaries(&ADM_metric, &metric_aux);

    // Read in primitive variables from gridfunctions
    primitive_quantities prims;
    ghl_initialize_primitives(
             auxevol_gfs[IDX4ptS(RHOBGF, index)],
             auxevol_gfs[IDX4ptS(PGF, index)],
             0.0/0.0, /*griddata.gridfuncs.auxevol_gfs[IDX4ptS(EPSILONGF, index)],*/
             auxevol_gfs[IDX4ptS(VU0GF, index)],
             auxevol_gfs[IDX4ptS(VU1GF, index)],
             auxevol_gfs[IDX4ptS(VU2GF, index)],
             0.0, /*auxevol_gfs[IDX4ptS(BU0GF, index)],*/
             0.0, /*auxevol_gfs[IDX4ptS(BU1GF, index)],*/
             0.0, /*auxevol_gfs[IDX4ptS(BU2GF, index)],*/
          poison, poison, poison, &prims);
          //entropy[index], Y_e[index], temp[index],

          double const rho = prims.rho;
          double const P   = prims.press;

          // Step 2: Compute P_cold and eps_cold
          double P_cold, eps_cold;
          ghl_hybrid_compute_P_cold_and_eps_cold(eos, rho, &P_cold, &eps_cold);

          // Step 3: Set Gamma cold
          int polytropic_index = ghl_hybrid_find_polytropic_index(eos, rho);
          double const Gamma = eos->Gamma_ppoly[polytropic_index];

          // Step 4: Compute the derivative of cold pressure w.r.t. density,
          //   dP/drho = Gamma K rho^(Gamma-1)
          //           = Gamma/rho (K rho^Gamma)
          //           = Gamma P_cold/rho .
          double const dPcold_drho = Gamma * P_cold / rho;

          // Step 5: Compute eps_thermal
          double const eps_th = (P-P_cold)/( (eos->Gamma_th-1)*rho );

          // Step 6: Compute eps
          double const eps = eps_cold + eps_th;
          prims.eps = eps;

        // Read in conservative variables from gridfunctions
        conservative_quantities cons, cons_orig;
        ghl_initialize_conservatives(
              evol_gfs[IDX4ptS(RHO_STARGF, index)],
              evol_gfs[IDX4ptS(TAU_TILDEGF, index)],
              evol_gfs[IDX4ptS(STILDED0GF, index)],
              evol_gfs[IDX4ptS(STILDED1GF, index)],
              evol_gfs[IDX4ptS(STILDED2GF, index)],
              poison, poison, &cons);
              //Y_e[index], entropy[index],

    // Here we save the original values of conservative variables in cons_orig for debugging purposes.
    cons_orig = cons;

    //FIXME: might slow down the code. Was formerly a CCTK_WARN
    if(isnan(cons.rho*cons.tau*cons.SD[0]*cons.SD[1]*cons.SD[2])) {
      printf("NaN found at start of C2P kernel: index %d %d %d, rho_* = %e, ~tau = %e, ~S_i = %e %e %e, gij = %e %e %e %e %e %e, sqrt_detgamma = %e\n",
                 i, j, k, cons.rho, cons.tau, cons.SD[0], cons.SD[1], cons.SD[2],
                 ADM_metric.gammaDD[0][0], ADM_metric.gammaDD[0][1], ADM_metric.gammaDD[0][2], ADM_metric.gammaDD[1][1], ADM_metric.gammaDD[1][2], ADM_metric.gammaDD[2][2], ADM_metric.sqrt_detgamma);
      diagnostics.nan_found++;
    }

    /************* Main conservative-to-primitive logic ************/
    int check=0;
    if(cons.rho>0.0) {
      // Apply the tau floor
      if( eos->eos_type == ghl_eos_hybrid )
        ghl_apply_conservative_limits(
              ghl_params, eos, &ADM_metric,
              &prims, &cons, &diagnostics);

      // declare some variables for the C2P routine.
      conservative_quantities cons_undens;

      // Set the conserved variables required by the con2prim routine
      ghl_undensitize_conservatives(ADM_metric.sqrt_detgamma, &cons, &cons_undens);

      /************* Conservative-to-primitive recovery ************/
      int check = ghl_con2prim_multi_method(
            ghl_params, eos, &ADM_metric, &metric_aux,
            &cons_undens, &prims, &diagnostics);

      if(check==0) {
        //Check for NAN!
        if( isnan(prims.rho*prims.press*prims.eps*prims.vU[0]*prims.vU[1]*prims.vU[2]) ) {
          printf("***********************************************************\n"
                      "NAN found in function %s (file: %s)\n"
                      "Input conserved variables:\n"
                      "rho_*, ~tau, ~S_{i}: %e %e %e %e %e\n"
                      "Undensitized conserved variables:\n"
                      "D, tau, S_{i}: %e %e %e %e %e\n"
                      "Output primitive variables:\n"
                      "rho, P: %e %e\n"
                      "v: %e %e %e\n"
                      "***********************************************************",
                      __func__,__FILE__,
                      cons.rho, cons.tau, cons.SD[0], cons.SD[1], cons.SD[2],
                      cons_undens.rho, cons_undens.tau, cons_undens.SD[0], cons_undens.SD[1], cons_undens.SD[2],
                      prims.rho, prims.press,
                      prims.vU[0], prims.vU[1], prims.vU[2]);
        }
      } else {
        printf("Con2Prim and Font fix failed!");
        printf("diagnostics->failure_checker = %d rho_* = %e, ~tau = %e, ~S_i = %e %e %e, gij = %e %e %e %e %e %e, Psi6 = %e",
                diagnostics.failure_checker, cons_orig.rho, cons_orig.tau, cons_orig.SD[0], cons_orig.SD[1], cons_orig.SD[2],
                ADM_metric.gammaDD[0][0], ADM_metric.gammaDD[0][1], ADM_metric.gammaDD[0][2], ADM_metric.gammaDD[1][1], ADM_metric.gammaDD[1][2], ADM_metric.gammaDD[2][2], ADM_metric.sqrt_detgamma);
      }
    } else {
      diagnostics.failure_checker+=1;
      ghl_set_prims_to_constant_atm(eos, &prims);
      rho_star_fix_applied++;
    } // if rho_star>0
    /***************************************************************/

    if( check != 0 ) {
      //--------------------------------------------------
      //----------- Primitive recovery failed ------------
      //--------------------------------------------------
      // Sigh, reset to atmosphere
      ghl_set_prims_to_constant_atm(eos, &prims);
      diagnostics.failure_checker+=100000;
      atm_resets++;
      // Then flag this point as a "success"
      check = 0;
      printf("Couldn't find root from: %e %e %e %e %e, rhob approx=%e, rho_b_atm=%e, gij=%e %e %e %e %e %e, alpha=%e\n",
                 cons_orig.rho, cons_orig.tau, cons_orig.SD[0], cons_orig.SD[1], cons_orig.SD[2], cons_orig.rho/ADM_metric.sqrt_detgamma, eos->rho_atm,
                 ADM_metric.gammaDD[0][0], ADM_metric.gammaDD[0][1], ADM_metric.gammaDD[0][2], ADM_metric.gammaDD[1][1], ADM_metric.gammaDD[1][2], ADM_metric.gammaDD[2][2], ADM_metric.lapse);
    }

    //--------------------------------------------------
    //---------- Primitive recovery succeeded ----------
    //--------------------------------------------------
    // Enforce limits on primitive variables and recompute conservatives.
    stress_energy Tmunu;
    ghl_enforce_primitive_limits_and_compute_u0(
          ghl_params, eos, &ADM_metric,
          &prims, &diagnostics.failure_checker);
    ghl_compute_conservs(
          &ADM_metric, &metric_aux, &prims, &cons);

    //Now we compute the difference between original & new conservatives, for diagnostic purposes:
    error_int_numer += fabs(cons.tau - cons_orig.tau) + fabs(cons.rho - cons_orig.rho) + fabs(cons.SD[0] - cons_orig.SD[0])
                       + fabs(cons.SD[1] - cons_orig.SD[1]) + fabs(cons.SD[2] - cons_orig.SD[2]);
    error_int_denom += cons_orig.tau + cons_orig.rho + fabs(cons_orig.SD[0]) + fabs(cons_orig.SD[1]) + fabs(cons_orig.SD[2]);

    if(check!=0) {
      diagnostics.failures++;
      if(ADM_metric.sqrt_detgamma > ghl_params->psi6threshold) {
        failures_inhoriz++;
        pointcount_inhoriz++;
      }
    }

    ghl_return_primitives(&prims,
             &auxevol_gfs[IDX4ptS(RHOBGF, index)],
             &auxevol_gfs[IDX4ptS(PGF, index)],
             &dummy0,
             &auxevol_gfs[IDX4ptS(VU0GF, index)],
             &auxevol_gfs[IDX4ptS(VU1GF, index)],
             &auxevol_gfs[IDX4ptS(VU2GF, index)],
             &dummy1,
             &dummy2,
             &dummy3,
             &dummy4, &dummy5, &dummy6);
          //&entropy[index], &Y_e[index], &temp[index]);

    ghl_return_conservatives(&cons,
          &evol_gfs[IDX4ptS(RHO_STARGF, index)],
          &evol_gfs[IDX4ptS(TAU_TILDEGF, index)],
          &evol_gfs[IDX4ptS(STILDED0GF, index)],
          &evol_gfs[IDX4ptS(STILDED1GF, index)],
          &evol_gfs[IDX4ptS(STILDED2GF, index)],
          &dummy1, &dummy2);
          //&Y_e[index], &entropy[index]);

    pointcount++;
    failures += diagnostics.failures;
    vel_limited_ptcount += diagnostics.speed_limited;
    backup0 += diagnostics.backup[0];
    backup1 += diagnostics.backup[1];
    backup2 += diagnostics.backup[2];
    nan_found += diagnostics.nan_found;
    n_iter += diagnostics.n_iter;
    }

  printf("C2P: NumPts= %d | Fixes: %d VL= %d rho*= %d | Failures: %d InHoriz= %d / %d | Error: %.3e, ErrDenom: %.3e | %.2f iters/gridpt\n",
             pointcount, backup0,
             vel_limited_ptcount, rho_star_fix_applied,
             failures, failures_inhoriz, pointcount_inhoriz,
             error_int_numer/error_int_denom, error_int_denom,
             (double)n_iter/( (double)(Nxx_plus_2NGHOSTS0*Nxx_plus_2NGHOSTS1*Nxx_plus_2NGHOSTS2) ));
  """

    add_to_Cfunction_dict(
                includes=includes,
                desc=desc,
                name=name, params=params,
                body=body,
                loopopts = "")

In [12]:
def add_to_Cfunction_dict__copy_and_outflow_bcs():
    name = "copy_and_outflow_bcs"
    desc = "Apply copy BCs to density and pressure, and outflow BCs to velocities, i.e. no inflowing material"
    includes = ["NRPy_basic_defines.h"]
    params = "const paramstruct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict auxevol_gfs"

    body = r"""

  // Unpack bc_info from bcstruct
  const bc_info_struct *bc_info = &bcstruct->bc_info;

  const int NUM_PRIM_GFs = 5;
  const int prims_gfs[5] = {RHOBGF, PGF, VU0GF, VU1GF, VU2GF};

  ////////////////////////////////////////////////////////
  // STEP 1 of 2: Apply BCs to pure outer boundary points.
  //              By "pure" we mean that these points are
  //              on the outer boundary and not also on
  //              an inner boundary.
  //              Here we fill in the innermost ghost zone
  //              layer first and move outward. At each
  //              layer, we fill in +/- x0 faces first,
  //              then +/- x1 faces, finally +/- x2 faces,
  //              filling in the edges as we go.
  // Spawn N OpenMP threads, either across all cores, or according to e.g., taskset.
  #pragma omp parallel
  {
    for(int which_gz=0;which_gz<NGHOSTS-1;which_gz++) for(int dirn=0;dirn<3;dirn++) {
        // This option results in about 1.6% slower runtime for SW curvilinear at 64x24x24 on 8-core Ryzen 9 4900HS
        //#pragma omp for collapse(2)
        //for(int which_gf=0;which_gf<NUM_EVOL_GFS;which_gf++) for(int idx2d=0;idx2d<bc_info->num_pure_outer_boundary_points[which_gz][dirn];idx2d++) {
        //  {
        // Don't spawn a thread if there are no boundary points to fill; results in a nice little speedup.
      if(bc_info->num_pure_outer_boundary_points[which_gz][dirn] > 0) {
#pragma omp for  // threads have been spawned; here we distribute across them
        for(int idx2d=0;idx2d<bc_info->num_pure_outer_boundary_points[which_gz][dirn];idx2d++) {
          const short i0 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].i0;
          const short i1 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].i1;
          const short i2 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].i2;
          const short FACEX0 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].FACEX0;
          const short FACEX1 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].FACEX1;
          const short FACEX2 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].FACEX2;
          const int idx = IDX3S(i0,i1,i2);
          const int idx_offset1 = IDX3S(i0+1*FACEX0,i1+1*FACEX1,i2+1*FACEX2);
          for(int which_gf=0;which_gf<NUM_PRIM_GFs;which_gf++) {
            // *** Apply copy BCs to all outer boundary points for prims GFs. ***
            auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx)] =
            auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx_offset1)];
          }
        }
      }
    }
  }

#pragma omp parallel
  {
    for(int which_gz=NGHOSTS-1;which_gz<NGHOSTS;which_gz++) for(int dirn=0;dirn<3;dirn++) {
        // This option results in about 1.6% slower runtime for SW curvilinear at 64x24x24 on 8-core Ryzen 9 4900HS
        //#pragma omp for collapse(2)
        //for(int which_gf=0;which_gf<NUM_EVOL_GFS;which_gf++) for(int idx2d=0;idx2d<bc_info->num_pure_outer_boundary_points[which_gz][dirn];idx2d++) {
        //  {
        // Don't spawn a thread if there are no boundary points to fill; results in a nice little speedup.
      if(bc_info->num_pure_outer_boundary_points[which_gz][dirn] > 0) {
#pragma omp for  // threads have been spawned; here we distribute across them
        for(int idx2d=0;idx2d<bc_info->num_pure_outer_boundary_points[which_gz][dirn];idx2d++) {
          const short i0 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].i0;
          const short i1 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].i1;
          const short i2 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].i2;
          const short FACEX0 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].FACEX0;
          const short FACEX1 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].FACEX1;
          const short FACEX2 = bcstruct->pure_outer_bc_array[dirn + (3*which_gz)][idx2d].FACEX2;
          const int idx = IDX3S(i0,i1,i2);
          const int idx_offset1 = IDX3S(i0+1*FACEX0,i1+1*FACEX1,i2+1*FACEX2);
          for(int which_gf=2;which_gf<NUM_PRIM_GFs;which_gf++) {
            // *** Apply outflow BCs to all outer boundary points for velocity GFs. ***
            auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx)] =
              0.5*(fabs(auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx)]) +
                        auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx)]);
          }
          for(int which_gf=0;which_gf<2;which_gf++) {
            // *** Apply copy to all outer boundary points for density and pressure GFs. ***
            auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx)] =
            auxevol_gfs[IDX4ptS(prims_gfs[which_gf], idx_offset1)];
          }
        }
      }
    }
  }

  ///////////////////////////////////////////////////////
  // STEP 2 of 2: Apply BCs to inner boundary points.
  //              These map to either the grid interior
  //              ("pure inner") or to pure outer boundary
  //              points ("inner maps to outer"). Those
  //              that map to outer require that outer be
  //              populated first; hence this being
  //              STEP 2 OF 2.
  // apply_bcs_inner_only(params, bcstruct, auxevol_gfs);
"""
    add_to_Cfunction_dict(
                includes=includes,
                desc=desc,
                name=name, params=params,
                body=body,
                loopopts = "")

<a id='mol'></a>

## Step 1.b: Generate Method of Lines timestepping code \[Back to [top](#toc)\]
$$\label{mol}$$

The Method of Lines algorithm is described in detail in the [**NRPy+ tutorial notebook on Method of Lines algorithm**](Tutorial-Method_of_Lines-C_Code_Generation.ipynb).

In [13]:
# Step 10: Generate Runge-Kutta-based (RK-based) timestepping code.
#       As described above the Table of Contents, this is a 3-step process:
#       10.A: Evaluate RHSs (RHS_string)
#       10.B: Apply boundary conditions (post_RHS_string, pt 1)
#       10.C: Enforce det(gammabar) = det(gammahat) constraint (post_RHS_string, pt 2)
import MoLtimestepping.MoL as MoL
# from MoLtimestepping.RK_Butcher_Table_Dictionary import Butcher_dict
# RK_order  = Butcher_dict[RK_method][1]
RHS_string      = """
grhd_rhs_eval(params, &griddata->eos, RK_INPUT_GFS, auxevol_gfs, RK_OUTPUT_GFS);"""

post_RHS_string = r"""
cons_to_prims(params, &griddata->ghl_params, &griddata->eos, RK_OUTPUT_GFS, auxevol_gfs);
copy_and_outflow_bcs(params, bcstruct, auxevol_gfs);"""

if not enable_rfm_precompute:
    RHS_string = RHS_string.replace("rfmstruct", "xx")
    post_RHS_string = post_RHS_string.replace("rfmstruct", "xx")

MoL.register_C_functions_and_NRPy_basic_defines(RK_method,
        RHS_string=RHS_string, post_RHS_string=post_RHS_string,
        enable_rfm=enable_rfm_precompute, enable_curviBCs=True, enable_SIMD=False)

<a id='OuterBoundaryConditions'></a>

## Step 1.b: Output needed C code for boundary condition driver \[Back to [top](#toc)\]
$$\label{OuterBoundaryConditions}$$

In [14]:
import CurviBoundaryConditions.CurviBoundaryConditions as CBC
CBC.CurviBoundaryConditions_register_NRPy_basic_defines()
CBC.CurviBoundaryConditions_register_C_functions(radiation_BC_FD_order=radiation_BC_FD_order)

Evolved gridfunction "AD0" has parity type 1.
Evolved gridfunction "AD1" has parity type 2.
Evolved gridfunction "AD2" has parity type 3.
Evolved gridfunction "StildeD0" has parity type 1.
Evolved gridfunction "StildeD1" has parity type 2.
Evolved gridfunction "StildeD2" has parity type 3.
Evolved gridfunction "phitilde" has parity type 0.
Evolved gridfunction "rho_star" has parity type 0.
Evolved gridfunction "tau_tilde" has parity type 0.
AuxEvol gridfunction "B_centerU0" has parity type 1.
AuxEvol gridfunction "B_centerU1" has parity type 2.
AuxEvol gridfunction "B_centerU2" has parity type 3.
AuxEvol gridfunction "B_staggerU0" has parity type 1.
AuxEvol gridfunction "B_staggerU1" has parity type 2.
AuxEvol gridfunction "B_staggerU2" has parity type 3.
AuxEvol gridfunction "B_staggerlU0" has parity type 1.
AuxEvol gridfunction "B_staggerlU1" has parity type 2.
AuxEvol gridfunction "B_staggerlU2" has parity type 3.
AuxEvol gridfunction "B_staggerrU0" has parity type 1.
AuxEvol gridfu

<a id='cparams_rfm_and_domainsize'></a>

## Step 1.c: Output C codes needed for declaring and setting Cparameters; also set `free_parameters.h` \[Back to [top](#toc)\]
$$\label{cparams_rfm_and_domainsize}$$

Here we output `free_parameters.h`, which sets initial data parameters, as well as grid domain & reference metric parameters, applying `domain_size` and `sinh_width`/`SymTP_bScale` (if applicable) as set above

In [15]:
# Step 3.e.i: Set free_parameters.h
outstr = r"""// Set free-parameter values.

// Set the default CFL Factor. Can be overwritten at command line.
REAL CFL_FACTOR = """+str(CFL_FACTOR)+";"


# Append to $Ccodesrootdir/free_parameters.h reference metric parameters based on generic
#    domain_size,sinh_width,sinhv2_const_dr,SymTP_bScale,
#    parameters set above.
outstr += rfm.out_default_free_parameters_for_rfm("returnstring",
                                                  domain_size,sinh_width,sinhv2_const_dr,SymTP_bScale)
with open(os.path.join(Ccodesrootdir,"free_parameters.h"),"w") as file:
    file.write(outstr.replace("params.", "griddata.params."))

<a id='cfl'></a>

## Step 1.d: Output needed C code for finding the minimum proper distance between grid points, needed for [CFL](https://en.wikipedia.org/w/index.php?title=Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition&oldid=806430673)-limited timestep \[Back to [top](#toc)\]
$$\label{cfl}$$

In order for our explicit-timestepping numerical solution to the scalar wave equation to be stable, it must satisfy the [CFL](https://en.wikipedia.org/w/index.php?title=Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition&oldid=806430673) condition:
$$
\Delta t \le \frac{\min(ds_i)}{c},
$$
where $c$ is the wavespeed, and
$$ds_i = h_i \Delta x^i$$ 
is the proper distance between neighboring gridpoints in the $i$th direction (in 3D, there are 3 directions), $h_i$ is the $i$th reference metric scale factor, and $\Delta x^i$ is the uniform grid spacing in the $i$th direction:

In [16]:
# Generate & register C function set_Nxx_dxx_invdx_params__and__xx()
# Generate & register C function xx_to_Cart() for
#               (the mapping from xx->Cartesian) for the chosen
#               CoordSystem:
# Generate & register the find_timestep() function

# Sets reference_metric globals: NRPy_basic_defines_str, rfm_struct__malloc, rfm_struct__define, rfm_struct__freemem
if enable_rfm_precompute:
    par.set_parval_from_str("reference_metric::rfm_precompute_Ccode_outdir", rfm_precompute_Ccode_outdir)
    par.set_parval_from_str("reference_metric::enable_rfm_precompute", "True")
    par.set_parval_from_str("reference_metric::rfm_precompute_to_Cfunctions_and_NRPy_basic_defines", "True")
    rfm.reference_metric()

rfm.register_C_functions(enable_rfm_precompute=enable_rfm_precompute, use_unit_wavespeed_for_find_timestep=True)
rfm.register_NRPy_basic_defines(enable_rfm_precompute=enable_rfm_precompute)

if enable_rfm_precompute:
    par.set_parval_from_str("reference_metric::enable_rfm_precompute", "False")
    rfm.ref_metric__hatted_quantities()

<a id='mainc'></a>

# Step 2: The C code `main()` function for `ScalarWaveCurvilinear_Playground` \[Back to [top](#toc)\]
$$\label{mainc}$$

Just as in [the start-to-finish, solving the scalar wave equation in Cartesian coordinates module](Tutorial-Start_to_Finish-ScalarWave.ipynb), we will implement the scalar wave equation via the Method of Lines. As discussed above, the critical differences between this code and the Cartesian version are as follows:
1. The CFL-constrained timestep depends on the proper distance between neighboring gridpoints
1. The boundary conditions must account for the fact that ghost zone points lying in the domain exterior can map either to the interior of the domain, or lie on the outer boundary. In the former case, we simply copy the data from the interior. In the latter case, we apply the usual outer boundary conditions.
1. The numerical grids must be staggered to avoid direct evaluation of the equations on coordinate singularities.

In [17]:
def add_to_Cfunction_dict_main__ghl_NRPy_IGM():
    includes = ["NRPy_basic_defines.h", "NRPy_function_prototypes.h", "time.h"]

    desc = """// main() function:
// Step 0: Read command-line input, set up grid structure, allocate memory for gridfunctions, set up coordinates
// Step 1: Write test data to gridfunctions
// Step 2: Overwrite all data in ghost zones with NaNs
// Step 3: Apply boundary conditions
// Step 4: Print gridfunction data after curvilinear boundary conditions have been applied
// Step 5: Free all allocated memory
"""
    c_type = "int"
    name = "main"
    params = "int argc, const char *argv[]"
    body = r"""

  griddata_struct griddata;

 /* ################################################
                Beginning of Initialize GRHayL
 ################################################ */

  // This section sets up the initial parameters that would normally
  // be provided by the simulation.
  const int backup_routine[3] = {Font1D,None,None};
  const bool calc_prims_guess = true;
  const double Psi6threshold = 1e100; //Taken from magnetizedTOV.par
  //const int update_Tmunu = 0; //IGM default = 1

  griddata.eos.eos_type = ghl_eos_hybrid;
  const int neos = 1;
  const double W_max = 10.0; // TPJ set this
  const double rho_b_min = 1.292852735094440e-10;
  const double rho_b_max = 1e300; //IGM default
  const double Gamma_th = 5.0/3.0; //Taken from magnetizedTOV.par
  const double rho_ppoly[1] = {0.0};
  const double Gamma_ppoly[1] = {5.0/3.0};
  const double k_ppoly0 = 1.0;
  const double Lorenz_damping_factor = 0.0/0.0;

  // Here, we initialize the structs that are (usually) static during
  // a simulation.

  ghl_initialize_params(Noble2D, backup_routine, false /*evolve entropy*/, false /*evolve temperature*/, calc_prims_guess, Psi6threshold, true /*Cupp Fix*/, Lorenz_damping_factor, &griddata.ghl_params);

  ghl_initialize_hybrid_eos_functions_and_params(W_max,
                                             rho_b_min, rho_b_min, rho_b_max,
                                             neos, rho_ppoly, Gamma_ppoly,
                                             k_ppoly0, Gamma_th, &griddata.eos);

    /* ################################################
                     End of Initialize GRHayL
     ################################################ */

  set_Cparameters_to_default(&griddata.params);

  // Step 0a: Read command-line input, error out if nonconformant
  if(argc != 5 || atoi(argv[1]) < NGHOSTS || atoi(argv[2]) < NGHOSTS || atoi(argv[3]) < NGHOSTS) {
    printf("Error: Expected one command-line argument: ./ScalarWaveCurvilinear_Playground Nx0 Nx1 Nx2,\n");
    printf("where Nx[0,1,2] is the number of grid points in the 0, 1, and 2 directions.\n");
    printf("Nx[] MUST BE larger than NGHOSTS (= %d)\n",NGHOSTS);
    exit(1);
  }
  // Step 0b: Set up numerical grid structure, first in space...
  const int Nxx[3] = { atoi(argv[1]), atoi(argv[2]), atoi(argv[3]) };
  // if(Nxx[0]%2 != 0 || Nxx[1]%2 != 0 || Nxx[2]%2 != 0) {
  //   printf("Error: Cannot guarantee a proper cell-centered grid if number of grid cells not set to even number.\n");
  //   printf("       For example, in case of angular directions, proper symmetry zones will not exist.\n");
  //   exit(1);
  // }

  // Step 0c: Set free parameters, overwriting Cparameters defaults
  //          by hand or with command-line input, as desired.
#include "free_parameters.h"

  // Step 0d: Uniform coordinate grids are stored to *xx[3]
  // Step 0d.i: Set bcstruct
  {
    int EigenCoord;
    EigenCoord = 1;
    // Step 0d.ii: Call set_Nxx_dxx_invdx_params__and__xx(), which sets
    //             params Nxx,Nxx_plus_2NGHOSTS,dxx,invdx, and xx[] for the
    //             chosen Eigen-CoordSystem.
    set_Nxx_dxx_invdx_params__and__xx(EigenCoord, Nxx, &griddata.params, griddata.xx);
    // Step 0e: Find ghostzone mappings; set up bcstruct
    bcstruct_set_up(&griddata.params, griddata.xx, &griddata.bcstruct);
    // Step 0e.i: Free allocated space for xx[][] array
    for(int i=0;i<3;i++) free(griddata.xx[i]);

    // Step 0f: Call set_Nxx_dxx_invdx_params__and__xx(), which sets
    //          params Nxx,Nxx_plus_2NGHOSTS,dxx,invdx, and xx[] for the
    //          chosen (non-Eigen) CoordSystem.
    EigenCoord = 0;
    set_Nxx_dxx_invdx_params__and__xx(EigenCoord, Nxx, &griddata.params, griddata.xx);
  }

  // Step 0g: Time coordinate parameters
   const REAL t_final = 0.4; //0.55;
   /* Final time is set so that at t=t_final,
                                          * data at the origin have not been corrupted
                                          * by the approximate outer boundary condition */

  // Step 0h: Set timestep based on smallest proper distance between gridpoints and CFL factor
  REAL dt = find_timestep(&griddata.params, griddata.xx, CFL_FACTOR);
  griddata.params.dt = dt;
  //printf("# Timestep set to = %e\n",(double)dt);
  int N_final = (int)(t_final / dt + 0.5); // The number of points in time.
                                           // Add 0.5 to account for C rounding down
                                           // typecasts to integers.
  // printf("Will take %d iterations\n", N_final);

  int output_every_N = (int)((REAL)N_final/100.0);
  if(output_every_N == 0) output_every_N = 1;

  // Step 0i: Error out if the number of auxiliary gridfunctions outnumber evolved gridfunctions.
  //              This is a limitation of the RK method. You are always welcome to declare & allocate
  //              additional gridfunctions by hand.
  if(NUM_AUX_GFS > NUM_EVOL_GFS) {
    printf("Error: NUM_AUX_GFS > NUM_EVOL_GFS. Either reduce the number of auxiliary gridfunctions,\n");
    printf("       or allocate (malloc) by hand storage for *diagnostic_output_gfs. \n");
    exit(1);
  }

  // Step 0j: Declare struct for gridfunctions and allocate memory for y_n_gfs gridfunctions
  MoL_malloc_y_n_gfs(&griddata.params, &griddata.gridfuncs);

  // Step 0.l: Set up initial data to be exact solution at time=0:
  //griddata.params.time = 0.0; //exact_solution_all_points(&griddata.params, griddata.xx, griddata.gridfuncs.y_n_gfs);

  // Step 0.m: Allocate memory for non y_n_gfs. We do this here to free up
  //         memory for setting up initial data (for cases in which initial
  //         data setup is memory intensive.)
  MoL_malloc_non_y_n_gfs(&griddata.params, &griddata.gridfuncs);

  const int Nxx_plus_2NGHOSTS0 = griddata.params.Nxx_plus_2NGHOSTS0;
  const int Nxx_plus_2NGHOSTS1 = griddata.params.Nxx_plus_2NGHOSTS1;
  const int Nxx_plus_2NGHOSTS2 = griddata.params.Nxx_plus_2NGHOSTS2;
  const int Nxx_plus_2NGHOSTS_tot = Nxx_plus_2NGHOSTS0*Nxx_plus_2NGHOSTS1*Nxx_plus_2NGHOSTS2;

  REAL time = 0.0;
  // Step 0.l: Set up TOV initial data
  const char *initial_data_option = argv[4];
  initial_data(initial_data_option, &griddata.params, griddata.xx, griddata.gridfuncs.y_n_gfs, griddata.gridfuncs.auxevol_gfs);

  for(int ijk=0; ijk<Nxx_plus_2NGHOSTS_tot; ijk++) {

    primitive_quantities prims;
    metric_quantities metric;
    conservative_quantities cons;
    con2prim_diagnostics diagnostics;
    ghl_initialize_diagnostics(&diagnostics);
    double dummy0, dummy1, dummy2, dummy3, dummy4, dummy5, dummy6;

    ghl_initialize_primitives(
             griddata.gridfuncs.auxevol_gfs[IDX4ptS(RHOBGF, ijk)],
             griddata.gridfuncs.auxevol_gfs[IDX4ptS(PGF, ijk)],
             0.0, /*griddata.gridfuncs.auxevol_gfs[IDX4ptS(EPSILONGF, ijk)],*/
             griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU0GF, ijk)],
             griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU1GF, ijk)],
             griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU2GF, ijk)],
             0.0, /*auxevol_gfs[IDX4ptS(BU0GF, index)],*/
             0.0, /*auxevol_gfs[IDX4ptS(BU1GF, index)],*/
             0.0, /*auxevol_gfs[IDX4ptS(BU2GF, index)],*/
             0.0,
             0.0,
             0.0,
             &prims);

     ghl_initialize_metric(
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(ALPHAGF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(BETAU0GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(BETAU1GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(BETAU2GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(GAMMADD00GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(GAMMADD01GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(GAMMADD02GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(GAMMADD11GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(GAMMADD12GF, ijk)],
        griddata.gridfuncs.auxevol_gfs[IDX4ptS(GAMMADD22GF, ijk)],
          &metric);

    ADM_aux_quantities metric_aux;
    ghl_compute_ADM_auxiliaries(&metric, &metric_aux);

   int speed_limited = 0;
   //This applies inequality fixes on the conservatives
    ghl_enforce_primitive_limits_and_compute_u0(
          &griddata.ghl_params, &griddata.eos, &metric,
          &prims, &speed_limited);

    //This computes the conservatives and stress-energy tensor from the new primitives
    ghl_compute_conservs(&metric, &metric_aux, &prims, &cons);

    griddata.gridfuncs.auxevol_gfs[IDX4ptS(U4UTGF, ijk)] = prims.u0;

    ghl_return_primitives(&prims,
             &griddata.gridfuncs.auxevol_gfs[IDX4ptS(RHOBGF, ijk)],
             &griddata.gridfuncs.auxevol_gfs[IDX4ptS(PGF, ijk)],
             &dummy0,
             &griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU0GF, ijk)],
             &griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU1GF, ijk)],
             &griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU2GF, ijk)],
             &dummy1,
             &dummy2,
             &dummy3,
             &dummy4, &dummy5, &dummy6);

   ghl_return_conservatives(&cons,
          &(griddata.gridfuncs.y_n_gfs[IDX4ptS(RHO_STARGF, ijk)]),
          &(griddata.gridfuncs.y_n_gfs[IDX4ptS(TAU_TILDEGF, ijk)]),
          &(griddata.gridfuncs.y_n_gfs[IDX4ptS(STILDED0GF, ijk)]),
          &(griddata.gridfuncs.y_n_gfs[IDX4ptS(STILDED1GF, ijk)]),
          &(griddata.gridfuncs.y_n_gfs[IDX4ptS(STILDED2GF, ijk)]),
          &dummy1, &dummy2);
  }

   cons_to_prims(&griddata.params, &griddata.ghl_params,
                 &griddata.eos,
                 griddata.gridfuncs.y_n_gfs,
                 griddata.gridfuncs.auxevol_gfs);

  for(int n=0;n<=N_final;n++)
  { // Main loop to progress forward in time.

  // Step 1: Set current time to correct value & compute exact solution
  time = ((REAL)n)*dt;

  // Step 2c: Output 1D slice of data.
  char filename[100];
  sprintf(filename,"output/out_rhob_z_%d_iter_%d.txt", Nxx_plus_2NGHOSTS0, n);
  FILE *out1_zD = fopen(filename, "w");
  const int i2 = (int)(Nxx_plus_2NGHOSTS2/2);
  const int i1 = (int)(Nxx_plus_2NGHOSTS1/2);
  for (int i0 = 0; i0 < Nxx_plus_2NGHOSTS0; i0++) {
    const int idx = IDX3S(i0,i1,i2);
    fprintf(out1_zD,"%.16e %.16e %.16e %.16e %.16e %.16e %.16e %.16e %.16e\n",
            time,
            griddata.xx[0][i0], griddata.xx[1][i1], griddata.xx[2][i2],
            griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU0GF,idx)],
            griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU1GF,idx)],
            griddata.gridfuncs.auxevol_gfs[IDX4ptS(VU2GF,idx)],
            griddata.gridfuncs.auxevol_gfs[IDX4ptS(RHOBGF,idx)],
            griddata.gridfuncs.auxevol_gfs[IDX4ptS(PGF,idx)]);
    }
  fclose(out1_zD);

    // Step 3: Evolve initial data forward in time using Method of Lines with RK4 algorithm,
    //         applying outer boundary conditions.
    // Step 3.b: Step forward one timestep (t -> t+dt) in time using
    //           chosen RK-like MoL timestepping algorithm
    MoL_step_forward_in_time(&griddata);

} // End main loop to progress forward in time.

  // Step 4: Free all allocated memory
  free(griddata.bcstruct.inner_bc_array);
  for(int ng=0;ng<NGHOSTS*3;ng++) free(griddata.bcstruct.pure_outer_bc_array[ng]);
  MoL_free_memory_y_n_gfs(&griddata.params, &griddata.gridfuncs);
  MoL_free_memory_non_y_n_gfs(&griddata.params, &griddata.gridfuncs);
  for(int i=0;i<3;i++) free(griddata.xx[i]);
  return 0;
"""

    add_to_Cfunction_dict(
        includes=includes,
        desc=desc,
        c_type=c_type, name=name, params=params,
        body=body,
        rel_path_to_Cparams=os.path.join("."), enableCparameters=False)

Next, register all remaining C functions in `outC_function_dict`, and output `finite_difference_functions.h`. Also construct `NRPy_basic_defines.h`.

In [18]:
# Register all functions:
add_to_Cfunction_dict_initial_data_driver()
add_to_Cfunction_dict_ID_options()
add_to_Cfunction_dict_calculate_metric_gfs()
add_to_Cfunction_dict__metric_face_interpolation()
add_to_Cfunction_dict__reconstruction_loop()
add_to_Cfunction_dict__grhd_rhs_eval()
add_to_Cfunction_dict__conservs_to_prims()
add_to_Cfunction_dict__compute_B_and_Bstagger_from_A()
add_to_Cfunction_dict__copy_and_outflow_bcs()
add_to_Cfunction_dict_main__ghl_NRPy_IGM()

# #diagnostics
# import diagnostics_generic.process_2D_data as plot2D  # NRPy+: analysis of output data
# import diagnostics_generic.output_yz_or_xy_plane as planar_diags  # NRPy+: C code for generating output data

# list_of_outputs = ["y_n_gfs[IDX4ptS(CFGF,idx)]",
#                    "log10(fabs(diagnostic_output_gfs[IDX4ptS(HGF,idx)]))",
#                    "log10(fabs(griddata->gridfuncs.auxevol_gfs[IDX4S(RHOBGF, i0, i1, i2)]))",
#                    "log10(fabs(griddata->gridfuncs.auxevol_gfs[IDX4S(PGF, i0, i1, i2)]))"]
# planar_diags.add_to_Cfunction_dict__plane_diagnostics(plane="yz", include_ghosts=True,
#                                                       list_of_outputs=list_of_outputs, num_sig_figs=15)


In [19]:
import outputC as outC
outC.outputC_register_C_functions_and_NRPy_basic_defines(addl_includes=["ghl.h",
                                                                        "reconstruction.h",
                                                                        "flux_source.h",
                                                                        "atmosphere.h",
                                                                        "con2prim.h"])  # #define M_PI, etc.  # #define M_PI, etc.
# Declare paramstruct, register set_Cparameters_to_default(),
#   and output declare_Cparameters_struct.h and set_Cparameters[].h:
outC.NRPy_param_funcs_register_C_functions_and_NRPy_basic_defines(os.path.join(Ccodesrootdir))
par.register_NRPy_basic_defines()  # add `paramstruct params` to griddata struct.

gri.register_C_functions_and_NRPy_basic_defines(list_of_extras_in_griddata_struct=["eos_parameters eos",
                                                                                   "ghl_parameters ghl_params"])  # #define IDX3S(), etc.
fin.register_C_functions_and_NRPy_basic_defines(NGHOSTS_account_for_onezone_upwind=True,
                                                enable_SIMD=enable_SIMD)  # #define NGHOSTS, and UPWIND() macro if SIMD disabled

# Output functions for computing all finite-difference stencils.
#   Must be called after defining all functions depending on FD stencils.
if enable_FD_functions:
    fin.output_finite_difference_functions_h(path=Ccodesrootdir)

# Call this last: Set up NRPy_basic_defines.h and NRPy_function_prototypes.h.
outC.construct_NRPy_basic_defines_h(Ccodesrootdir, enable_SIMD=enable_SIMD)
outC.construct_NRPy_function_prototypes_h(Ccodesrootdir)

# outC_function_dict["set_Nxx_dxx_invdx_params__and__xx"] = outC_function_dict["set_Nxx_dxx_invdx_params__and__xx"].replace("+ (1.0/2.0)","+ (0.0/2.0)")

<a id='compileexec'></a>

# Step 3: Compile generated C codes & solve the scalar wave equation \[Back to [top](#toc)\]
$$\label{compileexec}$$

To aid in the cross-platform-compatible (with Windows, MacOS, & Linux) compilation and execution, we make use of `cmdline_helper` [(**Tutorial**)](Tutorial-cmdline_helper.ipynb).

In [20]:
import subprocess
import multiprocessing

repo_url = 'https://github.com/GRHayL/GRHayL.git'  # Replace with the actual repository URL

os.chdir(Ccodesrootdir)

result = subprocess.run(['git', 'clone', repo_url], capture_output=True, text=True)
print(result.stdout)

os.chdir("GRHayL")
# result = subprocess.run(['git', 'checkout', "scupp/Renaming"], capture_output=True, text=True)
# print(result.stdout)

result = subprocess.run(['./configure', '--prefix=./'], capture_output=True, text=True)
print(result.stdout)

cpus = str(int(multiprocessing.cpu_count()))

result = subprocess.run(['make', '-j'+cpus], capture_output=True, text=True)
print(result.stdout)

result = subprocess.run(['make', 'install'], capture_output=True, text=True)
print(result.stdout)

os.chdir("../../")


Beginning configuration script for GRHayL...
Detecting system... Linux
Checking if command grep exists... YES (/bin/grep)
Checking if command awk exists... YES (/usr/bin/awk)
Checking if build type is valid... YES (buildtype: debug-opt; build flags: -Wall -std=c99 -march=native -fno-finite-math-only -O2 -g)
Checking if cflags were provided... NO
Checking if clibs were provided... NO
Checking if compiler gcc exists... YES (/usr/bin/gcc)
Checking if C compiler works... YES (gcc -O0 <objs>)
Checking if compiler works with -lm linker flag... YES (gcc -O0 <objs> -lm)
Checking if compiler works with build flags... YES (gcc -Wall -std=c99 -march=native -fno-finite-math-only -O2 -g  <objs> -lm)
Checking if compiler works with -fopenmp flag... YES (gcc -O0 -fopenmp <objs> -lm)
Checking if command pkg-config exists... YES (/usr/bin/pkg-config)
Checking if hdf5 library is available with pkg-config... YES
Checking if compiler works with hdf5... YES (gcc -Wall -std=c99 -march=native -fno-finite-ma

In [21]:
import cmdline_helper as cmd

# Step 10.d.i: Set GRHayL directory
ghl_INC_FLAG = "-IGRHayL/include/grhayl/"
ghl_LIB_FLAG = "-LGRHayL/lib"

# Step 10.d.iii: Compile the code
cmd.new_C_compile(Ccodesrootdir,"GRHayL_NRPy_IGM_standalone",
                  uses_free_parameters_h=True,
                  compiler_opt_option="fast",
                  addl_CFLAGS=[ghl_INC_FLAG],
                  addl_libraries=[ghl_LIB_FLAG+" -lgrhayl"])

os.chdir(Ccodesrootdir)

# set path to GRHayL library
os.environ['LD_LIBRARY_PATH'] = 'GRHayL/lib/'

cmd.Execute("GRHayL_NRPy_IGM_standalone", "800 4 4 b1",
            file_to_redirect_stdout="output/out100.txt")

(EXEC): Executing `make -j14`...
MoL_step_forward_in_time.c: In function ‘MoL_step_forward_in_time’:
MoL_step_forward_in_time.c:23:20: warning: variable ‘xx’ set but not used [-Wunused-but-set-variable]
     REAL *restrict xx[3]; for(int ww=0;ww<3;ww++) xx[ww] = griddata->xx[ww];
                    ^~
MoL_step_forward_in_time.c:53:20: warning: variable ‘xx’ set but not used [-Wunused-but-set-variable]
     REAL *restrict xx[3]; for(int ww=0;ww<3;ww++) xx[ww] = griddata->xx[ww];
                    ^~
MoL_step_forward_in_time.c:84:20: warning: variable ‘xx’ set but not used [-Wunused-but-set-variable]
     REAL *restrict xx[3]; for(int ww=0;ww<3;ww++) xx[ww] = griddata->xx[ww];
                    ^~
MoL_step_forward_in_time.c:115:20: warning: variable ‘xx’ set but not used [-Wunused-but-set-variable]
     REAL *restrict xx[3]; for(int ww=0;ww<3;ww++) xx[ww] = griddata->xx[ww];
                    ^~
(BENCH): Finished executing in 1.42 seconds.
Finished compilation.
(EXEC): Executing `

KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = "white"
plt.rcParams['axes.facecolor'] = "white"
plt.rcParams['figure.dpi'] = "1000"


In [ ]:
raw_data = np.loadtxt(os.path.join("output", "out_rhob_z_808_iter_385.txt"))

plt.scatter(raw_data[:, 1], raw_data[:, -2], s=0.05)
plt.show()

<a id='latex_pdf_output'></a>

# Step 5: Output this notebook to $\LaTeX$-formatted PDF file \[Back to [top](#toc)\]
$$\label{latex_pdf_output}$$

The following code cell converts this Jupyter notebook into a proper, clickable $\LaTeX$-formatted PDF file. After the cell is successfully run, the generated PDF may be found in the root NRPy+ tutorial directory, with filename
[Tutorial-Start_to_Finish-ScalarWaveCurvilinear.pdf](Tutorial-Start_to_Finish-ScalarWaveCurvilinear.pdf) (Note that clicking on this link may not work; you may need to open the PDF file through another means.)

In [ ]:
import cmdline_helper as cmd    # NRPy+: Multi-platform Python command-line interface
cmd.output_Jupyter_notebook_to_LaTeXed_PDF("Tutorial-Start_to_Finish-ScalarWaveCurvilinear")